In [1]:
# ============================================================
# CELL 1 — Environment Setup + Full Data Inspection
# Notebook: 05_LSTM_AE_Training.ipynb
#
# PURPOSE: Verify every file we will touch before writing
# any model code. Nothing gets built until this passes.
#
# CHECKPOINT STRATEGY:
#   All training results save to: BASE/LSTM_AE/checkpoints/
#   If Colab resets, run Cell 1 + Cell 2 (resume cell).
#   You never rerun training from scratch.
# ============================================================

import os
import numpy as np
from google.colab import drive

# ── Mount Drive ───────────────────────────────────────────
drive.mount('/content/drive', force_remount=False)

# ── Base paths ────────────────────────────────────────────
BASE       = '/content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs'

WESAD_DIR  = os.path.join(BASE, 'WESAD')
AROAD_DIR  = os.path.join(BASE, 'AffectiveROAD')
DALIA_DIR  = os.path.join(BASE, 'PPGDaLiA')
EMW_DIR    = os.path.join(BASE, 'EmoWear')

LSTM_DIR        = os.path.join(BASE, 'LSTM_AE')
CHECKPOINT_DIR  = os.path.join(LSTM_DIR, 'checkpoints')
RESULTS_DIR     = os.path.join(LSTM_DIR, 'results')
MODELS_DIR      = os.path.join(LSTM_DIR, 'models')

for d in [CHECKPOINT_DIR, RESULTS_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f'  Ready: {d}')

# ── Locked constants ──────────────────────────────────────
FEATURE_NAMES = [
    'mean_HR', 'mean_RR', 'SDNN', 'RMSSD', 'pNN50',
    'mean_BR', 'std_BR',
    'mean_temp', 'std_temp',
    'mean_acc_mag', 'std_acc_mag'
]
N_FEATURES = len(FEATURE_NAMES)   # must be 11
T          = 5                     # sequence length, locked

# Active subject lists — hardcoded, never auto-detect
WESAD_SUBJECTS  = ['S2','S3','S4','S5','S6','S7','S8','S9',
                   'S10','S11','S13','S14','S15','S16','S17']
AROAD_DRIVES    = ['Drv1','Drv2','Drv3','Drv4','Drv5','Drv6','Drv7',
                   'Drv8','Drv9','Drv10','Drv11','Drv12','Drv13']
DALIA_SUBJECTS  = ['S1','S2','S3','S4','S5','S7','S9','S11','S13','S14','S15']
# S6,S8,S10,S12 excluded permanently (age/hardware)

# WESAD subjects excluded from stress evaluation (baseline LOSO only)
WESAD_STRESS_EXCLUDED = ['S3', 'S6']

print(f'\nFeature count  : {N_FEATURES}')
print(f'Sequence length: T={T}')
print(f'WESAD subjects : {len(WESAD_SUBJECTS)}')
print(f'AROAD drives   : {len(AROAD_DRIVES)}')
print(f'DaLiA subjects : {len(DALIA_SUBJECTS)}')

# ============================================================
# PART 1: Load all combined arrays and verify
# ============================================================
print('\n' + '='*60)
print('PART 1 — Load and verify all combined arrays')
print('='*60)

all_checks_passed = True

# ── WESAD ─────────────────────────────────────────────────
# WESAD is already normalised and pre-split by label
# No X_all / y_all — baseline and stress are separate files
print('\n--- WESAD ---')
try:
    W_base  = np.load(os.path.join(WESAD_DIR, 'combined', 'WESAD_all_baseline.npy'))
    W_stress= np.load(os.path.join(WESAD_DIR, 'combined', 'WESAD_all_stress.npy'))
    W_ids_b = np.load(os.path.join(WESAD_DIR, 'combined', 'WESAD_subject_labels_base.npy'),
                      allow_pickle=True)
    W_ids_s = np.load(os.path.join(WESAD_DIR, 'combined', 'WESAD_subject_labels_stress.npy'),
                      allow_pickle=True)

    print(f'  baseline shape : {W_base.shape}  dtype={W_base.dtype}')
    print(f'  stress shape   : {W_stress.shape}  dtype={W_stress.dtype}')
    print(f'  ids_base shape : {W_ids_b.shape}  sample={W_ids_b[:3]}')
    print(f'  ids_stress shape:{W_ids_s.shape}  sample={W_ids_s[:3]}')

    checks = {
        'base shape (566,11)' : W_base.shape  == (566, 11),
        'stress shape (271,11)': W_stress.shape== (271, 11),
        'ids_base (566,)'     : W_ids_b.shape == (566,),
        'ids_stress (271,)'   : W_ids_s.shape == (271,),
        'no NaN base'         : not np.isnan(W_base).any(),
        'no NaN stress'       : not np.isnan(W_stress).any(),
        'base features == 11' : W_base.shape[1] == N_FEATURES,
    }
    for k, v in checks.items():
        print(f'  {"✅" if v else "❌"} {k}')
        if not v: all_checks_passed = False

except FileNotFoundError as e:
    print(f'  ❌ FILE NOT FOUND: {e}')
    all_checks_passed = False

# ── AffectiveROAD ─────────────────────────────────────────
# Raw combined array with labels and drive IDs
print('\n--- AffectiveROAD ---')
try:
    AR_X   = np.load(os.path.join(AROAD_DIR, 'combined', 'AR_X_all_raw.npy'))
    AR_y   = np.load(os.path.join(AROAD_DIR, 'combined', 'AR_y_all.npy'))
    AR_ids = np.load(os.path.join(AROAD_DIR, 'combined', 'AR_drive_ids.npy'),
                     allow_pickle=True)

    n_base   = int(np.sum(AR_y == 0))
    n_stress = int(np.sum(AR_y == 1))
    print(f'  X shape   : {AR_X.shape}  dtype={AR_X.dtype}')
    print(f'  y shape   : {AR_y.shape}  dtype={AR_y.dtype}')
    print(f'  ids shape : {AR_ids.shape}  sample={AR_ids[:3]}')
    print(f'  baseline  : {n_base} windows')
    print(f'  stress    : {n_stress} windows')

    checks = {
        'X shape (1371,11)'   : AR_X.shape   == (1371, 11),
        'y shape (1371,)'     : AR_y.shape   == (1371,),
        'baseline == 716'     : n_base       == 716,
        'stress == 655'       : n_stress     == 655,
        'no NaN'              : not np.isnan(AR_X).any(),
        'features == 11'      : AR_X.shape[1]== N_FEATURES,
    }
    for k, v in checks.items():
        print(f'  {"✅" if v else "❌"} {k}')
        if not v: all_checks_passed = False

except FileNotFoundError as e:
    print(f'  ❌ FILE NOT FOUND: {e}')
    all_checks_passed = False

# ── PPG-DaLiA ─────────────────────────────────────────────
print('\n--- PPG-DaLiA ---')
try:
    DA_X   = np.load(os.path.join(DALIA_DIR, 'combined', 'DALIA_X_all_raw.npy'))
    DA_y   = np.load(os.path.join(DALIA_DIR, 'combined', 'DALIA_y_all.npy'))
    DA_ids = np.load(os.path.join(DALIA_DIR, 'combined', 'DALIA_subject_ids.npy'),
                     allow_pickle=True)

    print(f'  X shape   : {DA_X.shape}  dtype={DA_X.dtype}')
    print(f'  y shape   : {DA_y.shape}  dtype={DA_y.dtype}')
    print(f'  ids shape : {DA_ids.shape}  sample={DA_ids[:3]}')
    print(f'  unique y  : {np.unique(DA_y)} (should be [0] only)')

    checks = {
        'X shape (846,11)'    : DA_X.shape   == (846, 11),
        'y shape (846,)'      : DA_y.shape   == (846,),
        'all labels == 0'     : np.all(DA_y  == 0),
        'no NaN'              : not np.isnan(DA_X).any(),
        'features == 11'      : DA_X.shape[1]== N_FEATURES,
    }
    for k, v in checks.items():
        print(f'  {"✅" if v else "❌"} {k}')
        if not v: all_checks_passed = False

except FileNotFoundError as e:
    print(f'  ❌ FILE NOT FOUND: {e}')
    all_checks_passed = False

# ── EmoWear ───────────────────────────────────────────────
print('\n--- EmoWear ---')
try:
    EM_X   = np.load(os.path.join(EMW_DIR, 'combined', 'EMW_X_all_raw.npy'))
    EM_y   = np.load(os.path.join(EMW_DIR, 'combined', 'EMW_y_all.npy'))
    EM_ids = np.load(os.path.join(EMW_DIR, 'combined', 'EMW_subject_ids.npy'),
                     allow_pickle=True)

    print(f'  X shape   : {EM_X.shape}  dtype={EM_X.dtype}  (expect float32)')
    print(f'  y shape   : {EM_y.shape}  dtype={EM_y.dtype}')
    print(f'  ids shape : {EM_ids.shape}  sample={EM_ids[:3]}')
    print(f'  unique y  : {np.unique(EM_y)} (should be [0] only)')

    checks = {
        'X shape (125,11)'    : EM_X.shape   == (125, 11),
        'y shape (125,)'      : EM_y.shape   == (125,),
        'all labels == 0'     : np.all(EM_y  == 0),
        'no NaN'              : not np.isnan(EM_X.astype(np.float64)).any(),
        'features == 11'      : EM_X.shape[1]== N_FEATURES,
        'dtype float32'       : EM_X.dtype   == np.float32,
    }
    for k, v in checks.items():
        print(f'  {"✅" if v else "❌"} {k}')
        if not v: all_checks_passed = False

except FileNotFoundError as e:
    print(f'  ❌ FILE NOT FOUND: {e}')
    all_checks_passed = False

# ============================================================
# PART 2: Norm stats spot check — one subject per dataset
# ============================================================
print('\n' + '='*60)
print('PART 2 — Norm stats spot check')
print('='*60)

spot_checks = [
    # (label, path_to_mean, path_to_std)
    ('WESAD S2',
     os.path.join(WESAD_DIR, 'per_subject', 'S2_norm_params_mean.npy'),
     os.path.join(WESAD_DIR, 'per_subject', 'S2_norm_params_std.npy')),

    ('AffectiveROAD Drv1',
     os.path.join(AROAD_DIR, 'per_drive', 'AR_Drv1_norm_mean.npy'),
     os.path.join(AROAD_DIR, 'per_drive', 'AR_Drv1_norm_std.npy')),

    ('PPG-DaLiA S1',
     os.path.join(DALIA_DIR, 'per_subject', 'DALIA_S1_norm_mean.npy'),
     os.path.join(DALIA_DIR, 'per_subject', 'DALIA_S1_norm_std.npy')),

    ('EmoWear 01-9TZK',
     os.path.join(EMW_DIR, 'per_subject', 'EMW_01_norm_mean.npy'),
     os.path.join(EMW_DIR, 'per_subject', 'EMW_01_norm_std.npy')),
]

for label, mean_path, std_path in spot_checks:
    print(f'\n  {label}')
    try:
        mn = np.load(mean_path).astype(np.float64)
        st = np.load(std_path).astype(np.float64)
        print(f'    mean shape : {mn.shape}')
        print(f'    std  shape : {st.shape}')
        print(f'    mean values: {np.round(mn, 3)}')
        print(f'    std  values: {np.round(st, 3)}')
        if np.any(st == 0):
            print(f'    ❌ WARNING: zero std — division by zero in normalisation')
            all_checks_passed = False
        elif np.any(st < 0):
            print(f'    ❌ WARNING: negative std — impossible')
            all_checks_passed = False
        else:
            print(f'    ✅ std clean (no zeros, no negatives)')
    except FileNotFoundError:
        print(f'    ❌ FILE NOT FOUND: {mean_path}')
        print(f'    Trying to list what is actually in that folder...')
        folder = os.path.dirname(mean_path)
        if os.path.exists(folder):
            files = sorted(os.listdir(folder))
            print(f'    Files found ({len(files)} total):')
            for f in files[:10]:
                print(f'      {f}')
        else:
            print(f'    Folder does not exist: {folder}')
        all_checks_passed = False

# ============================================================
# PART 3: Sequence count check at T=5 per subject
# ============================================================
print('\n' + '='*60)
print(f'PART 3 — Sequence counts at T={T} per subject')
print('='*60)

MIN_SEQ = 10   # flag anything below this

# WESAD
print(f'\n  WESAD (baseline only):')
for sid in WESAD_SUBJECTS:
    mask = W_ids_b == sid
    n_win = int(np.sum(mask))
    n_seq = max(0, n_win - T + 1)
    stress_note = ' [stress excluded from eval]' if sid in WESAD_STRESS_EXCLUDED else ''
    flag = ' ⚠️  LOW' if n_seq < MIN_SEQ else ''
    print(f'    {sid:4s}  windows={n_win:3d}  sequences={n_seq:3d}{flag}{stress_note}')

# AffectiveROAD
print(f'\n  AffectiveROAD (baseline only):')
for drv in AROAD_DRIVES:
    mask = (AR_ids == drv) & (AR_y == 0)
    n_win = int(np.sum(mask))
    n_seq = max(0, n_win - T + 1)
    flag = ' ⚠️  LOW' if n_seq < MIN_SEQ else ''
    print(f'    {drv:6s}  windows={n_win:3d}  sequences={n_seq:3d}{flag}')

# PPG-DaLiA
print(f'\n  PPG-DaLiA (all windows are baseline):')
for sid in DALIA_SUBJECTS:
    mask = DA_ids == sid
    n_win = int(np.sum(mask))
    n_seq = max(0, n_win - T + 1)
    flag = ' ⚠️  LOW' if n_seq < MIN_SEQ else ''
    print(f'    {sid:4s}  windows={n_win:3d}  sequences={n_seq:3d}{flag}')

# EmoWear
print(f'\n  EmoWear (external validation only — not LOSO):')
emw_subjects = np.unique(EM_ids)
print(f'    {len(emw_subjects)} active subjects, {len(EM_X)} total windows')
print(f'    ~{len(EM_X)//len(emw_subjects)} windows/subject — cannot form T={T} sequences')
print(f'    Role: global false alarm check after all 39 LOSO runs complete')

# ============================================================
# FINAL VERDICT
# ============================================================
print('\n' + '='*60)
if all_checks_passed:
    print('✅ ALL CHECKS PASSED — ready to build the model pipeline')
else:
    print('❌ SOME CHECKS FAILED — fix issues above before proceeding')
print('='*60)
print(f'\nTotal LOSO runs planned: {len(WESAD_SUBJECTS) + len(AROAD_DRIVES) + len(DALIA_SUBJECTS)}')
print(f'  WESAD        : {len(WESAD_SUBJECTS)} subjects')
print(f'  AffectiveROAD: {len(AROAD_DRIVES)} drives')
print(f'  PPG-DaLiA    : {len(DALIA_SUBJECTS)} subjects')
print(f'  EmoWear      : external check only (post LOSO)')

Mounted at /content/drive
  Ready: /content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs/LSTM_AE/checkpoints
  Ready: /content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs/LSTM_AE/results
  Ready: /content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs/LSTM_AE/models

Feature count  : 11
Sequence length: T=5
WESAD subjects : 15
AROAD drives   : 13
DaLiA subjects : 11

PART 1 — Load and verify all combined arrays

--- WESAD ---
  baseline shape : (566, 11)  dtype=float64
  stress shape   : (271, 11)  dtype=float64
  ids_base shape : (566,)  sample=['S2' 'S2' 'S2']
  ids_stress shape:(271,)  sample=['S2' 'S2' 'S2']
  ✅ base shape (566,11)
  ✅ stress shape (271,11)
  ✅ ids_base (566,)
  ✅ ids_stress (271,)
  ✅ no NaN base
  ✅ no NaN stress
  ✅ base features == 11

--- AffectiveROAD ---
  X shape   : (1371, 11)  dtype=float64
  y shape   : (1371,)  dtype=int64
  ids shape : (1371,)  sample=['Drv1' 'Drv1' 'Drv1']
  baseline  : 716 windows
  stress    : 655 windows
  ✅ X shape (1371,11)
  ✅

In [2]:
# ============================================================
# CELL 1b — EmoWear std=0 investigation
# Run immediately after Cell 1 before anything else.
# ============================================================

import os
import numpy as np

EMW_DIR = '/content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs/EmoWear'

print('Checking ALL EmoWear subjects for zero or near-zero std values')
print('(features 9=mean_acc_mag, 10=std_acc_mag are the suspects)')
print()

zero_std_subjects = []

# Load all per-subject norm stds and check
emw_subject_ids_path = os.path.join(EMW_DIR, 'combined', 'EMW_subject_ids.npy')
all_ids = np.load(emw_subject_ids_path, allow_pickle=True)
unique_ids = np.unique(all_ids)

for sid in unique_ids:
    # Convert full folder name to numeric prefix e.g. '01-9TZK' -> '01'
    numeric = sid.split('-')[0]
    std_path = os.path.join(EMW_DIR, 'per_subject', f'EMW_{numeric}_norm_std.npy')
    try:
        st = np.load(std_path).astype(np.float64)
        zero_mask = st == 0.0
        tiny_mask = (st > 0) & (st < 1e-6)
        if np.any(zero_mask) or np.any(tiny_mask):
            zero_std_subjects.append(sid)
            print(f'  {sid}: std = {np.round(st, 6)}')
            print(f'    zero features  : {np.where(zero_mask)[0].tolist()}')
            print(f'    tiny (<1e-6)   : {np.where(tiny_mask)[0].tolist()}')
    except FileNotFoundError:
        print(f'  {sid}: FILE NOT FOUND')

if not zero_std_subjects:
    print('No subjects have zero or near-zero std. All safe.')
else:
    print(f'\n{len(zero_std_subjects)} subjects have zero/tiny std values.')
    print('These features will be handled with a safe normalisation:')
    print('  (value - mean) / max(std, 1e-8)')
    print('This prevents division by zero while preserving the signal.')
    print('Since EmoWear is external validation only (not LOSO training),')
    print('this affects false alarm rate computation only.')

Checking ALL EmoWear subjects for zero or near-zero std values
(features 9=mean_acc_mag, 10=std_acc_mag are the suspects)

  06-9UA7: std = [1.830990e-01 1.285050e+00 3.042010e+00 1.557876e+00 0.000000e+00
 2.809772e+00 1.640173e+00 1.026220e-01 5.862400e-02 1.063000e-03
 1.450000e-04]
    zero features  : []
    tiny (<1e-6)   : [4]
  26-9VTJ: std = [1.158348e+00 8.902446e+00 6.295131e+00 8.006460e-01 0.000000e+00
 7.073690e-01 5.869840e-01 2.679200e-02 3.839000e-03 1.474000e-03
 2.960000e-04]
    zero features  : []
    tiny (<1e-6)   : [4]

2 subjects have zero/tiny std values.
These features will be handled with a safe normalisation:
  (value - mean) / max(std, 1e-8)
This prevents division by zero while preserving the signal.
Since EmoWear is external validation only (not LOSO training),
this affects false alarm rate computation only.


In [3]:
# ============================================================
# CELL 2 — Normalisation + Sequence Builder
# Run after Cell 1 passes all checks.
#
# WHAT THIS CELL DOES:
#   1. Normalises raw arrays using per-subject/drive stats
#   2. Builds sequences of shape (n_seq, T=5, 11) from
#      consecutive baseline windows per subject
#   3. Saves per-subject checkpoint files to Drive
#
# CHECKPOINT FILES SAVED:
#   LSTM_AE/checkpoints/WESAD_SN_processed.npz
#     → base_seqs: (n_seq, 5, 11)  normalised baseline sequences
#     → stress_wins: (n_stress, 11) normalised stress windows
#
#   LSTM_AE/checkpoints/AROAD_DrvN_processed.npz
#     → base_seqs: (n_seq, 5, 11)
#     → stress_wins: (n_stress, 11)
#
#   LSTM_AE/checkpoints/DALIA_SN_processed.npz
#     → base_seqs: (n_seq, 5, 11)
#
#   LSTM_AE/checkpoints/EMW_normalised.npy
#     → (125, 11) all EmoWear windows normalised, float64
#
#   LSTM_AE/checkpoints/cell2_summary.json
#     → record of what was saved, sequence counts, any issues
#
# RESUME LOGIC:
#   If Colab resets, this cell checks which .npz files already
#   exist and skips them. Only processes what is missing.
# ============================================================

import os
import json
import numpy as np
from datetime import datetime

# ── Paths (must match Cell 1) ─────────────────────────────
BASE          = '/content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs'
WESAD_DIR     = os.path.join(BASE, 'WESAD')
AROAD_DIR     = os.path.join(BASE, 'AffectiveROAD')
DALIA_DIR     = os.path.join(BASE, 'PPGDaLiA')
EMW_DIR       = os.path.join(BASE, 'EmoWear')
CHECKPOINT_DIR= os.path.join(BASE, 'LSTM_AE', 'checkpoints')

T          = 5
N_FEATURES = 11
WESAD_SUBJECTS       = ['S2','S3','S4','S5','S6','S7','S8','S9',
                        'S10','S11','S13','S14','S15','S16','S17']
WESAD_STRESS_EXCLUDED= ['S3', 'S6']
AROAD_DRIVES         = ['Drv1','Drv2','Drv3','Drv4','Drv5','Drv6','Drv7',
                        'Drv8','Drv9','Drv10','Drv11','Drv12','Drv13']
DALIA_SUBJECTS       = ['S1','S2','S3','S4','S5','S7','S9','S11','S13','S14','S15']

# ── Helper functions ──────────────────────────────────────

def build_sequences(windows, T):
    """
    Convert flat (n_windows, 11) array into overlapping sequences.
    Returns (n_windows - T + 1, T, 11).
    Each sequence is T consecutive windows.
    If n_windows < T, returns empty array with shape (0, T, 11).
    """
    n = len(windows)
    if n < T:
        return np.empty((0, T, N_FEATURES), dtype=np.float64)
    seqs = np.stack([windows[i:i+T] for i in range(n - T + 1)])
    return seqs.astype(np.float64)

def safe_normalise(X_raw, mean, std):
    """
    Z-score normalise using pre-computed per-subject stats.
    Uses max(std, 1e-8) to prevent division by zero.
    Handles float32 input by casting to float64.
    """
    X = X_raw.astype(np.float64)
    m = mean.astype(np.float64)
    s = np.maximum(std.astype(np.float64), 1e-8)
    return (X - m) / s

# ── Summary log ──────────────────────────────────────────
summary = {
    'generated'  : datetime.now().strftime('%Y-%m-%d %H:%M'),
    'T'          : T,
    'WESAD'      : {},
    'AffectiveROAD': {},
    'PPGDaLiA'   : {},
    'EmoWear'    : {},
    'issues'     : []
}

total_saved = 0
total_skipped = 0

# ============================================================
# WESAD
# Already normalised. Load per-subject files directly.
# Build sequences from baseline. Keep stress windows flat.
# ============================================================
print('='*60)
print('Processing WESAD...')
print('='*60)

for sid in WESAD_SUBJECTS:
    out_path = os.path.join(CHECKPOINT_DIR, f'WESAD_{sid}_processed.npz')

    if os.path.exists(out_path):
        print(f'  {sid}: already saved, skipping')
        total_skipped += 1
        continue

    try:
        # Load already-normalised per-subject files
        base_path   = os.path.join(WESAD_DIR, 'per_subject',
                                   f'{sid}_baseline_normalized.npy')
        base_wins   = np.load(base_path).astype(np.float64)

        # Stress: excluded subjects have no stress in eval pool
        if sid in WESAD_STRESS_EXCLUDED:
            stress_wins = np.empty((0, N_FEATURES), dtype=np.float64)
            stress_note = 'stress excluded from eval'
        else:
            stress_path = os.path.join(WESAD_DIR, 'per_subject',
                                       f'{sid}_stress_normalized.npy')
            stress_wins = np.load(stress_path).astype(np.float64)
            stress_note = ''

        # Build sequences from baseline
        base_seqs = build_sequences(base_wins, T)

        np.savez_compressed(out_path,
                            base_seqs   = base_seqs,
                            stress_wins = stress_wins)

        summary['WESAD'][sid] = {
            'base_windows'  : len(base_wins),
            'base_sequences': len(base_seqs),
            'stress_windows': len(stress_wins),
            'note'          : stress_note
        }
        total_saved += 1
        print(f'  {sid}: base_seqs={base_seqs.shape}  '
              f'stress_wins={stress_wins.shape}  '
              f'{"[" + stress_note + "]" if stress_note else ""}')

    except Exception as e:
        msg = f'WESAD {sid} FAILED: {e}'
        print(f'  ❌ {msg}')
        summary['issues'].append(msg)

# ============================================================
# AffectiveROAD
# Raw combined array. Normalise per drive using per_drive stats.
# Separate baseline and stress. Build sequences from baseline.
# ============================================================
print(f'\n{"="*60}')
print('Processing AffectiveROAD...')
print('='*60)

AR_X   = np.load(os.path.join(AROAD_DIR, 'combined', 'AR_X_all_raw.npy'))
AR_y   = np.load(os.path.join(AROAD_DIR, 'combined', 'AR_y_all.npy'))
AR_ids = np.load(os.path.join(AROAD_DIR, 'combined', 'AR_drive_ids.npy'),
                 allow_pickle=True)

for drv in AROAD_DRIVES:
    out_path = os.path.join(CHECKPOINT_DIR, f'AROAD_{drv}_processed.npz')

    if os.path.exists(out_path):
        print(f'  {drv}: already saved, skipping')
        total_skipped += 1
        continue

    try:
        # Extract this drive's windows
        mask     = AR_ids == drv
        X_drive  = AR_X[mask]
        y_drive  = AR_y[mask]

        # Load norm stats
        mean = np.load(os.path.join(AROAD_DIR, 'per_drive',
                                    f'AR_{drv}_norm_mean.npy'))
        std  = np.load(os.path.join(AROAD_DIR, 'per_drive',
                                    f'AR_{drv}_norm_std.npy'))

        # Normalise all windows using baseline stats
        X_norm = safe_normalise(X_drive, mean, std)

        # Separate baseline and stress
        base_wins   = X_norm[y_drive == 0]
        stress_wins = X_norm[y_drive == 1]

        # Build sequences from baseline
        base_seqs = build_sequences(base_wins, T)

        np.savez_compressed(out_path,
                            base_seqs   = base_seqs,
                            stress_wins = stress_wins)

        summary['AffectiveROAD'][drv] = {
            'base_windows'  : len(base_wins),
            'base_sequences': len(base_seqs),
            'stress_windows': len(stress_wins),
        }
        total_saved += 1
        print(f'  {drv}: base_seqs={base_seqs.shape}  '
              f'stress_wins={stress_wins.shape}')

    except Exception as e:
        msg = f'AROAD {drv} FAILED: {e}'
        print(f'  ❌ {msg}')
        summary['issues'].append(msg)

# ============================================================
# PPG-DaLiA
# Raw combined array. Normalise per subject. All baseline only.
# Build sequences.
# ============================================================
print(f'\n{"="*60}')
print('Processing PPG-DaLiA...')
print('='*60)

DA_X   = np.load(os.path.join(DALIA_DIR, 'combined', 'DALIA_X_all_raw.npy'))
DA_ids = np.load(os.path.join(DALIA_DIR, 'combined', 'DALIA_subject_ids.npy'),
                 allow_pickle=True)

for sid in DALIA_SUBJECTS:
    out_path = os.path.join(CHECKPOINT_DIR, f'DALIA_{sid}_processed.npz')

    if os.path.exists(out_path):
        print(f'  {sid}: already saved, skipping')
        total_skipped += 1
        continue

    try:
        # Extract this subject's windows
        mask   = DA_ids == sid
        X_subj = DA_X[mask]

        # Load norm stats
        mean = np.load(os.path.join(DALIA_DIR, 'per_subject',
                                    f'DALIA_{sid}_norm_mean.npy'))
        std  = np.load(os.path.join(DALIA_DIR, 'per_subject',
                                    f'DALIA_{sid}_norm_std.npy'))

        # Normalise
        X_norm = safe_normalise(X_subj, mean, std)

        # Build sequences (all baseline)
        base_seqs = build_sequences(X_norm, T)

        np.savez_compressed(out_path, base_seqs=base_seqs)

        summary['PPGDaLiA'][sid] = {
            'base_windows'  : len(X_norm),
            'base_sequences': len(base_seqs),
        }
        total_saved += 1
        print(f'  {sid}: base_seqs={base_seqs.shape}')

    except Exception as e:
        msg = f'DALIA {sid} FAILED: {e}'
        print(f'  ❌ {msg}')
        summary['issues'].append(msg)

# ============================================================
# EmoWear
# Raw combined array, float32. Normalise per subject with
# safe std (max(std, 1e-8)). Keep flat — cannot form sequences.
# Save one combined normalised array for external FAR check.
# ============================================================
print(f'\n{"="*60}')
print('Processing EmoWear...')
print('='*60)

emw_out_path = os.path.join(CHECKPOINT_DIR, 'EMW_normalised.npy')

if os.path.exists(emw_out_path):
    print('  EmoWear: already saved, skipping')
    total_skipped += 1
else:
    try:
        EM_X   = np.load(os.path.join(EMW_DIR, 'combined', 'EMW_X_all_raw.npy'))
        EM_ids = np.load(os.path.join(EMW_DIR, 'combined', 'EMW_subject_ids.npy'),
                         allow_pickle=True)

        # Normalise per subject, collect into one array
        X_norm_all = np.zeros((len(EM_X), N_FEATURES), dtype=np.float64)
        unique_ids = np.unique(EM_ids)
        issues_emw = []

        for sid in unique_ids:
            numeric  = sid.split('-')[0]
            mask     = EM_ids == sid
            X_subj   = EM_X[mask]

            mean = np.load(os.path.join(EMW_DIR, 'per_subject',
                                        f'EMW_{numeric}_norm_mean.npy'))
            std  = np.load(os.path.join(EMW_DIR, 'per_subject',
                                        f'EMW_{numeric}_norm_std.npy'))

            # safe_normalise already applies max(std, 1e-8)
            X_norm_all[mask] = safe_normalise(X_subj, mean, std)

        np.save(emw_out_path, X_norm_all)

        summary['EmoWear'] = {
            'total_windows'  : len(X_norm_all),
            'active_subjects': len(unique_ids),
            'role'           : 'external FAR check only, post LOSO',
            'note'           : 'float32 cast to float64, safe std applied'
        }
        total_saved += 1
        print(f'  Saved: {X_norm_all.shape}  dtype={X_norm_all.dtype}')
        print(f'  Subjects processed: {len(unique_ids)}')
        print(f'  NaN check: {np.isnan(X_norm_all).sum()} NaNs')

    except Exception as e:
        msg = f'EmoWear FAILED: {e}'
        print(f'  ❌ {msg}')
        summary['issues'].append(msg)

# ============================================================
# Save summary JSON
# ============================================================
summary_path = os.path.join(CHECKPOINT_DIR, 'cell2_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

# ============================================================
# Final report
# ============================================================
print(f'\n{"="*60}')
print('CELL 2 COMPLETE')
print('='*60)
print(f'  Files saved   : {total_saved}')
print(f'  Files skipped : {total_skipped} (already existed)')
print(f'  Issues        : {len(summary["issues"])}')

if summary['issues']:
    print('\n  ISSUES FOUND:')
    for issue in summary['issues']:
        print(f'    ❌ {issue}')
else:
    print('\n  No issues. All checkpoint files written successfully.')

print(f'\n  Checkpoint location:')
print(f'  {CHECKPOINT_DIR}')
print(f'\n  Files written:')
all_ckpt = sorted(os.listdir(CHECKPOINT_DIR))
for f in all_ckpt:
    size_kb = os.path.getsize(os.path.join(CHECKPOINT_DIR, f)) / 1024
    print(f'    {f:45s} {size_kb:6.1f} KB')

print(f'\n{"="*60}')
if not summary['issues']:
    print('✅ Ready for Cell 3 — LOSO training loop')
else:
    print('❌ Fix issues above before proceeding to Cell 3')
print('='*60)

Processing WESAD...
  S2: already saved, skipping
  S3: already saved, skipping
  S4: already saved, skipping
  S5: already saved, skipping
  S6: already saved, skipping
  S7: already saved, skipping
  S8: already saved, skipping
  S9: already saved, skipping
  S10: already saved, skipping
  S11: already saved, skipping
  S13: already saved, skipping
  S14: already saved, skipping
  S15: already saved, skipping
  S16: already saved, skipping
  S17: already saved, skipping

Processing AffectiveROAD...
  Drv1: already saved, skipping
  Drv2: already saved, skipping
  Drv3: already saved, skipping
  Drv4: already saved, skipping
  Drv5: already saved, skipping
  Drv6: already saved, skipping
  Drv7: already saved, skipping
  Drv8: already saved, skipping
  Drv9: already saved, skipping
  Drv10: already saved, skipping
  Drv11: already saved, skipping
  Drv12: already saved, skipping
  Drv13: already saved, skipping

Processing PPG-DaLiA...
  S1: already saved, skipping
  S2: already saved

In [4]:
# ============================================================
# CELL 3 — LSTM-AE Architecture + Smoke Test
#
# PART A: Define the model architecture
# PART B: Smoke test on WESAD S2 (train on all others,
#         test on S2 baseline + stress)
#
# Only proceed to Cell 4 (full LOSO loop) if smoke test
# shows: training loss decreasing, baseline recon error
# lower than stress recon error for S2.
# ============================================================

import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

# ── Paths ─────────────────────────────────────────────────
BASE           = '/content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs'
CHECKPOINT_DIR = os.path.join(BASE, 'LSTM_AE', 'checkpoints')
RESULTS_DIR    = os.path.join(BASE, 'LSTM_AE', 'results')
MODELS_DIR     = os.path.join(BASE, 'LSTM_AE', 'models')

# ── Locked constants ──────────────────────────────────────
T          = 5
N_FEATURES = 11
WESAD_SUBJECTS       = ['S2','S3','S4','S5','S6','S7','S8','S9',
                        'S10','S11','S13','S14','S15','S16','S17']
WESAD_STRESS_EXCLUDED= ['S3', 'S6']
AROAD_DRIVES         = ['Drv1','Drv2','Drv3','Drv4','Drv5','Drv6','Drv7',
                        'Drv8','Drv9','Drv10','Drv11','Drv12','Drv13']
DALIA_SUBJECTS       = ['S1','S2','S3','S4','S5','S7','S9','S11','S13','S14','S15']

# ── Device ────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ============================================================
# PART A — LSTM-AE Architecture
# ============================================================

class LSTMAutoEncoder(nn.Module):
    """
    LSTM-Based Autoencoder for self-supervised baseline modelling.

    Architecture:
      Encoder LSTM: (batch, T, 11) → hidden state (batch, hidden_size)
      Bottleneck:   hidden state of final encoder layer
      Decoder LSTM: repeated bottleneck (batch, T, hidden_size) → (batch, T, hidden_size)
      Output layer: linear projection (batch, T, hidden_size) → (batch, T, 11)

    Training objective: minimise MSE between input and reconstruction.
    Anomaly score at inference: MSE of reconstructed vs actual window.
    """
    def __init__(self, n_features=11, hidden_size=64, n_layers=1):
        super(LSTMAutoEncoder, self).__init__()

        self.n_features  = n_features
        self.hidden_size = hidden_size
        self.n_layers    = n_layers
        self.T           = T

        # Encoder: reads input sequence, compresses to hidden state
        self.encoder = nn.LSTM(
            input_size  = n_features,
            hidden_size = hidden_size,
            num_layers  = n_layers,
            batch_first = True
        )

        # Decoder: takes repeated bottleneck, reconstructs sequence
        self.decoder = nn.LSTM(
            input_size  = hidden_size,
            hidden_size = hidden_size,
            num_layers  = n_layers,
            batch_first = True
        )

        # Output projection: hidden_size → n_features
        self.output_layer = nn.Linear(hidden_size, n_features)

    def forward(self, x):
        # x: (batch, T, n_features)
        batch_size = x.size(0)

        # Encode → bottleneck is the final hidden state
        _, (h_n, _) = self.encoder(x)
        # h_n: (n_layers, batch, hidden_size)
        bottleneck = h_n[-1]  # (batch, hidden_size) — final layer only

        # Repeat bottleneck T times for decoder input
        decoder_input = bottleneck.unsqueeze(1).repeat(1, self.T, 1)
        # decoder_input: (batch, T, hidden_size)

        # Decode
        decoder_out, _ = self.decoder(decoder_input)
        # decoder_out: (batch, T, hidden_size)

        # Project to feature space
        reconstruction = self.output_layer(decoder_out)
        # reconstruction: (batch, T, n_features)

        return reconstruction


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Instantiate and inspect
model_test = LSTMAutoEncoder(n_features=N_FEATURES, hidden_size=64, n_layers=1)
print(f'\nLSTM-AE Architecture:')
print(model_test)
print(f'\nTrainable parameters: {count_parameters(model_test):,}')

# Verify forward pass with correct shapes
dummy_input = torch.randn(8, T, N_FEATURES)   # batch=8, T=5, features=11
dummy_output = model_test(dummy_input)
print(f'\nShape check:')
print(f'  Input  : {dummy_input.shape}')
print(f'  Output : {dummy_output.shape}')
assert dummy_output.shape == dummy_input.shape, 'Output shape mismatch!'
print(f'  ✅ Input and output shapes match')

# ============================================================
# PART B — Helper functions used by smoke test and LOSO loop
# ============================================================

def train_model(model, train_seqs, epochs=50, batch_size=32, lr=1e-3, verbose=True):
    """
    Train LSTM-AE on baseline sequences.
    train_seqs: numpy array (n_sequences, T, n_features)
    Returns: trained model, list of per-epoch losses
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    # Build DataLoader
    X_tensor = torch.tensor(train_seqs, dtype=torch.float32)
    dataset  = TensorDataset(X_tensor)
    loader   = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model.train()
    losses = []

    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        for (batch,) in loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            recon = model(batch)
            loss  = criterion(recon, batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(batch)

        avg_loss = epoch_loss / len(train_seqs)
        losses.append(avg_loss)

        if verbose and (epoch % 10 == 0 or epoch == 1):
            print(f'    Epoch {epoch:3d}/{epochs}  loss={avg_loss:.6f}')

    return model, losses


def score_windows(model, windows, device):
    """
    Compute reconstruction error for individual windows.
    Each window (11,) is repeated T times → sequence (1, T, 11).
    Returns MSE per window as numpy array.

    Why repeated window approach:
    The model learned sequences of resting physiology. Feeding
    a single window repeated T times tests whether that window's
    feature pattern is consistent with the learned baseline.
    Baseline windows → low error. Stress windows → high error.
    """
    model.eval()
    scores = []
    with torch.no_grad():
        for w in windows:
            # w: (11,)
            seq = torch.tensor(w, dtype=torch.float32)
            seq = seq.unsqueeze(0).unsqueeze(0).repeat(1, T, 1)
            # seq: (1, T, 11)
            seq = seq.to(device)
            recon = model(seq)
            mse = torch.mean((seq - recon) ** 2).item()
            scores.append(mse)
    return np.array(scores)


def score_sequences(model, seqs, device):
    """
    Compute reconstruction error for actual sequences.
    seqs: (n_seq, T, 11)
    Returns mean MSE per sequence as numpy array.
    Used for threshold calibration on baseline sequences.
    """
    model.eval()
    scores = []
    with torch.no_grad():
        for seq in seqs:
            s = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)
            recon = model(s)
            mse = torch.mean((s - recon) ** 2).item()
            scores.append(mse)
    return np.array(scores)


def compute_threshold(baseline_scores, percentile=95):
    """
    Set anomaly threshold at the given percentile of baseline
    reconstruction errors. Windows above this are flagged anomalous.
    """
    return float(np.percentile(baseline_scores, percentile))


def compute_metrics(stress_scores, baseline_scores, threshold):
    """
    Compute AUROC, F1, precision, recall, false alarm rate.
    stress_scores  : reconstruction errors on stress windows
    baseline_scores: reconstruction errors on baseline windows (for FAR)
    threshold      : anomaly cutoff
    """
    # Build ground truth labels: 1=stress (anomaly), 0=baseline
    y_true   = np.concatenate([np.ones(len(stress_scores)),
                                np.zeros(len(baseline_scores))])
    y_scores = np.concatenate([stress_scores, baseline_scores])
    y_pred   = (y_scores > threshold).astype(int)

    auroc = float(roc_auc_score(y_true, y_scores))
    f1    = float(f1_score(y_true, y_pred, zero_division=0))
    prec  = float(precision_score(y_true, y_pred, zero_division=0))
    rec   = float(recall_score(y_true, y_pred, zero_division=0))

    # False alarm rate: fraction of baseline windows flagged as anomalous
    far = float(np.mean(y_pred[y_true == 0]))

    return {'AUROC': auroc, 'F1': f1, 'precision': prec,
            'recall': rec, 'FAR': far, 'threshold': threshold}


def load_checkpoint(path):
    return np.load(path, allow_pickle=True)


# ============================================================
# PART C — Smoke test on WESAD S2
# Train on everyone except S2, test on S2.
# ============================================================
print('\n' + '='*60)
print('SMOKE TEST — WESAD S2')
print('  Training on all subjects except S2')
print('  Testing on S2 baseline + stress')
print('='*60)

# Build training pool: all baseline sequences except S2
train_seqs_list = []

for sid in WESAD_SUBJECTS:
    if sid == 'S2':
        continue
    data = load_checkpoint(
        os.path.join(CHECKPOINT_DIR, f'WESAD_{sid}_processed.npz'))
    train_seqs_list.append(data['base_seqs'])

for drv in AROAD_DRIVES:
    data = load_checkpoint(
        os.path.join(CHECKPOINT_DIR, f'AROAD_{drv}_processed.npz'))
    train_seqs_list.append(data['base_seqs'])

for sid in DALIA_SUBJECTS:
    data = load_checkpoint(
        os.path.join(CHECKPOINT_DIR, f'DALIA_{sid}_processed.npz'))
    train_seqs_list.append(data['base_seqs'])

train_seqs = np.concatenate(train_seqs_list, axis=0)
print(f'\nTraining pool: {train_seqs.shape} sequences')
print(f'  ({len(train_seqs)} sequences x T={T} x {N_FEATURES} features)')

# Load S2 test data
s2_data      = load_checkpoint(
    os.path.join(CHECKPOINT_DIR, 'WESAD_S2_processed.npz'))
s2_base_seqs = s2_data['base_seqs']    # (33, 5, 11) — for threshold
s2_stress    = s2_data['stress_wins']  # (19, 11)    — for evaluation

# Extract baseline windows from sequences (last window of each seq)
# for per-window FAR scoring
s2_base_wins = s2_base_seqs[:, -1, :]  # (33, 11)

print(f'\nTest subject S2:')
print(f'  baseline sequences : {s2_base_seqs.shape}')
print(f'  stress windows     : {s2_stress.shape}')

# Train
print(f'\nTraining LSTM-AE (50 epochs)...')
model_smoke = LSTMAutoEncoder(n_features=N_FEATURES, hidden_size=64, n_layers=1)
model_smoke, losses = train_model(model_smoke, train_seqs,
                                  epochs=50, batch_size=32,
                                  lr=1e-3, verbose=True)

# Verify loss decreased
print(f'\nLoss trajectory:')
print(f'  Epoch  1: {losses[0]:.6f}')
print(f'  Epoch 25: {losses[24]:.6f}')
print(f'  Epoch 50: {losses[49]:.6f}')
if losses[-1] < losses[0]:
    print(f'  ✅ Loss decreased ({losses[0]:.6f} → {losses[-1]:.6f})')
else:
    print(f'  ❌ Loss did NOT decrease — something is wrong')

# Compute anomaly scores
print(f'\nComputing anomaly scores...')
base_seq_scores = score_sequences(model_smoke, s2_base_seqs, device)
stress_scores   = score_windows(model_smoke, s2_stress, device)
base_win_scores = score_windows(model_smoke, s2_base_wins, device)

threshold = compute_threshold(base_seq_scores, percentile=95)

print(f'\nReconstruction errors:')
print(f'  Baseline (sequences) mean : {base_seq_scores.mean():.6f}')
print(f'  Baseline (sequences) 95th : {threshold:.6f}  ← threshold')
print(f'  Stress   (windows)   mean : {stress_scores.mean():.6f}')
print(f'  Stress mean > Baseline mean: '
      f'{"✅ YES" if stress_scores.mean() > base_seq_scores.mean() else "❌ NO"}')

# Compute metrics
metrics = compute_metrics(stress_scores, base_win_scores, threshold)
print(f'\nSmoke test metrics for S2:')
for k, v in metrics.items():
    print(f'  {k:12s}: {v:.4f}')

print(f'\n{"="*60}')
if (losses[-1] < losses[0]) and (stress_scores.mean() > base_seq_scores.mean()):
    print('✅ SMOKE TEST PASSED — proceed to Cell 4 (full LOSO loop)')
else:
    print('❌ SMOKE TEST FAILED — investigate before running full LOSO')
print('='*60)

Device: cuda

LSTM-AE Architecture:
LSTMAutoEncoder(
  (encoder): LSTM(11, 64, batch_first=True)
  (decoder): LSTM(64, 64, batch_first=True)
  (output_layer): Linear(in_features=64, out_features=11, bias=True)
)

Trainable parameters: 53,707

Shape check:
  Input  : torch.Size([8, 5, 11])
  Output : torch.Size([8, 5, 11])
  ✅ Input and output shapes match

SMOKE TEST — WESAD S2
  Training on all subjects except S2
  Testing on S2 baseline + stress

Training pool: (1939, 5, 11) sequences
  (1939 sequences x T=5 x 11 features)

Test subject S2:
  baseline sequences : (33, 5, 11)
  stress windows     : (19, 11)

Training LSTM-AE (50 epochs)...
    Epoch   1/50  loss=0.752045
    Epoch  10/50  loss=0.223603
    Epoch  20/50  loss=0.141162
    Epoch  30/50  loss=0.101135
    Epoch  40/50  loss=0.079357
    Epoch  50/50  loss=0.067630

Loss trajectory:
  Epoch  1: 0.752045
  Epoch 25: 0.120307
  Epoch 50: 0.067630
  ✅ Loss decreased (0.752045 → 0.067630)

Computing anomaly scores...

Reconst

In [5]:
# ============================================================
# CELL 4 — Full LOSO Training Loop
#
# Runs 39 LOSO iterations:
#   15 WESAD subjects  → AUROC + F1 + precision + recall + FAR
#   13 AffectiveROAD   → AUROC + F1 + precision + recall + FAR
#   11 PPG-DaLiA       → FAR only (no stress labels)
#
# CHECKPOINT STRATEGY:
#   Results saved to LSTM_AE/results/ after EVERY run.
#   If Colab resets, re-run this cell. It detects completed
#   runs and skips them. Only processes what is missing.
#   You will never lose a completed run.
#
# OUTPUT FILES:
#   results/WESAD_SN_result.json     — per-subject metrics
#   results/AROAD_DrvN_result.json
#   results/DALIA_SN_result.json
#   results/loso_all_results.json    — combined after all done
# ============================================================

import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from datetime import datetime

# ── Paths ─────────────────────────────────────────────────
BASE           = '/content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs'
CHECKPOINT_DIR = os.path.join(BASE, 'LSTM_AE', 'checkpoints')
RESULTS_DIR    = os.path.join(BASE, 'LSTM_AE', 'results')
MODELS_DIR     = os.path.join(BASE, 'LSTM_AE', 'models')

# ── Constants ─────────────────────────────────────────────
T          = 5
N_FEATURES = 11
HIDDEN     = 64
N_LAYERS   = 1
EPOCHS     = 100
BATCH_SIZE = 32
LR         = 1e-3
THRESHOLD_PCT  = 95    # percentile for anomaly threshold
EARLY_STOP_PAT = 10   # stop if no improvement for this many epochs
EARLY_STOP_MIN = 0.001 # minimum improvement to count as progress

WESAD_SUBJECTS        = ['S2','S3','S4','S5','S6','S7','S8','S9',
                         'S10','S11','S13','S14','S15','S16','S17']
WESAD_STRESS_EXCLUDED = ['S3', 'S6']
AROAD_DRIVES          = ['Drv1','Drv2','Drv3','Drv4','Drv5','Drv6','Drv7',
                         'Drv8','Drv9','Drv10','Drv11','Drv12','Drv13']
DALIA_SUBJECTS        = ['S1','S2','S3','S4','S5','S7','S9','S11','S13','S14','S15']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Model definition (same as Cell 3) ─────────────────────
class LSTMAutoEncoder(nn.Module):
    def __init__(self, n_features=11, hidden_size=64, n_layers=1):
        super().__init__()
        self.T = T
        self.encoder = nn.LSTM(n_features, hidden_size,
                               n_layers, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, hidden_size,
                               n_layers, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, n_features)

    def forward(self, x):
        _, (h_n, _) = self.encoder(x)
        bottleneck   = h_n[-1]
        dec_in       = bottleneck.unsqueeze(1).repeat(1, self.T, 1)
        dec_out, _   = self.decoder(dec_in)
        return self.output_layer(dec_out)


# ── Training function with early stopping ─────────────────
def train_model(model, train_seqs, epochs, batch_size, lr,
                patience, min_delta):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    X_tensor = torch.tensor(train_seqs, dtype=torch.float32)
    loader   = DataLoader(TensorDataset(X_tensor),
                          batch_size=batch_size, shuffle=True)
    model.train()
    losses       = []
    best_loss    = float('inf')
    no_improve   = 0

    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        for (batch,) in loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch), batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(batch)

        avg = epoch_loss / len(train_seqs)
        losses.append(avg)

        if best_loss - avg > min_delta:
            best_loss  = avg
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            break   # early stopping

    return model, losses


# ── Scoring functions (same as Cell 3) ────────────────────
def score_windows(model, windows):
    model.eval()
    scores = []
    with torch.no_grad():
        for w in windows:
            seq = torch.tensor(w, dtype=torch.float32)
            seq = seq.unsqueeze(0).unsqueeze(0).repeat(1, T, 1).to(device)
            mse = torch.mean((seq - model(seq)) ** 2).item()
            scores.append(mse)
    return np.array(scores)


def score_sequences(model, seqs):
    model.eval()
    scores = []
    with torch.no_grad():
        for seq in seqs:
            s   = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)
            mse = torch.mean((s - model(s)) ** 2).item()
            scores.append(mse)
    return np.array(scores)


def compute_threshold(scores, pct):
    return float(np.percentile(scores, pct))


def compute_metrics(stress_scores, base_scores, threshold):
    y_true   = np.concatenate([np.ones(len(stress_scores)),
                                np.zeros(len(base_scores))])
    y_scores = np.concatenate([stress_scores, base_scores])
    y_pred   = (y_scores > threshold).astype(int)
    auroc = float(roc_auc_score(y_true, y_scores))
    return {
        'AUROC'    : auroc,
        'F1'       : float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall'   : float(recall_score(y_true, y_pred, zero_division=0)),
        'FAR'      : float(np.mean(y_pred[y_true == 0])),
        'threshold': threshold,
    }


def load_ckpt(name):
    return np.load(os.path.join(CHECKPOINT_DIR, name), allow_pickle=True)


# ── Build training pool excluding one subject ─────────────
def build_training_pool(exclude_dataset, exclude_id):
    """
    Returns numpy array of all baseline sequences except the
    test subject's sequences.
    exclude_dataset: 'WESAD', 'AROAD', or 'DALIA'
    exclude_id     : subject ID string e.g. 'S2' or 'Drv1'
    """
    seqs = []
    for sid in WESAD_SUBJECTS:
        if exclude_dataset == 'WESAD' and sid == exclude_id:
            continue
        seqs.append(load_ckpt(f'WESAD_{sid}_processed.npz')['base_seqs'])
    for drv in AROAD_DRIVES:
        if exclude_dataset == 'AROAD' and drv == exclude_id:
            continue
        seqs.append(load_ckpt(f'AROAD_{drv}_processed.npz')['base_seqs'])
    for sid in DALIA_SUBJECTS:
        if exclude_dataset == 'DALIA' and sid == exclude_id:
            continue
        seqs.append(load_ckpt(f'DALIA_{sid}_processed.npz')['base_seqs'])
    return np.concatenate(seqs, axis=0)


# ============================================================
# LOSO LOOP
# ============================================================

# Build the ordered list of all 39 runs
loso_runs = (
    [('WESAD',  sid) for sid in WESAD_SUBJECTS] +
    [('AROAD',  drv) for drv in AROAD_DRIVES]   +
    [('DALIA',  sid) for sid in DALIA_SUBJECTS]
)

total_runs     = len(loso_runs)
completed_runs = 0
skipped_runs   = 0

print(f'\nTotal LOSO runs: {total_runs}')
print(f'Max epochs per run: {EPOCHS} (early stopping patience={EARLY_STOP_PAT})')
print(f'Threshold percentile: {THRESHOLD_PCT}th\n')

for run_idx, (dataset, subject_id) in enumerate(loso_runs, 1):

    result_path = os.path.join(RESULTS_DIR,
                               f'{dataset}_{subject_id}_result.json')

    # ── Resume: skip if already done ──────────────────────
    if os.path.exists(result_path):
        print(f'[{run_idx:02d}/{total_runs}] {dataset} {subject_id}: '
              f'already done, skipping')
        skipped_runs += 1
        continue

    print(f'\n[{run_idx:02d}/{total_runs}] {dataset} {subject_id}')
    t_start = datetime.now()

    # ── Build training pool ────────────────────────────────
    train_seqs = build_training_pool(dataset, subject_id)
    print(f'  Train pool : {len(train_seqs)} sequences')

    # ── Train fresh model ──────────────────────────────────
    model = LSTMAutoEncoder(N_FEATURES, HIDDEN, N_LAYERS)
    model, losses = train_model(model, train_seqs,
                                EPOCHS, BATCH_SIZE, LR,
                                EARLY_STOP_PAT, EARLY_STOP_MIN)
    actual_epochs = len(losses)
    print(f'  Epochs run : {actual_epochs}  '
          f'(loss {losses[0]:.4f} → {losses[-1]:.4f})')

    # ── Load test subject data ─────────────────────────────
    if dataset == 'WESAD':
        ckpt         = load_ckpt(f'WESAD_{subject_id}_processed.npz')
        base_seqs    = ckpt['base_seqs']
        stress_wins  = ckpt['stress_wins']
        has_stress   = (subject_id not in WESAD_STRESS_EXCLUDED
                        and len(stress_wins) > 0)

    elif dataset == 'AROAD':
        ckpt         = load_ckpt(f'AROAD_{subject_id}_processed.npz')
        base_seqs    = ckpt['base_seqs']
        stress_wins  = ckpt['stress_wins']
        has_stress   = len(stress_wins) > 0

    else:  # DALIA
        ckpt         = load_ckpt(f'DALIA_{subject_id}_processed.npz')
        base_seqs    = ckpt['base_seqs']
        stress_wins  = np.empty((0, N_FEATURES))
        has_stress   = False

    # ── Score baseline sequences → threshold ──────────────
    base_seq_scores = score_sequences(model, base_seqs)
    threshold       = compute_threshold(base_seq_scores, THRESHOLD_PCT)

    # Baseline windows for FAR (use last window of each sequence)
    base_win_scores = score_windows(model, base_seqs[:, -1, :])
    far_base        = float(np.mean(base_win_scores > threshold))

    # ── Score stress windows and compute full metrics ──────
    if has_stress:
        stress_scores = score_windows(model, stress_wins)
        metrics       = compute_metrics(stress_scores,
                                        base_win_scores, threshold)
        print(f'  AUROC      : {metrics["AUROC"]:.4f}  '
              f'F1={metrics["F1"]:.4f}  '
              f'FAR={metrics["FAR"]:.4f}')
    else:
        stress_scores = np.array([])
        metrics       = {
            'AUROC': None, 'F1': None,
            'precision': None, 'recall': None,
            'FAR': far_base, 'threshold': threshold
        }
        reason = ('stress excluded from eval'
                  if dataset == 'WESAD' and subject_id in WESAD_STRESS_EXCLUDED
                  else 'no stress labels in dataset')
        print(f'  FAR        : {far_base:.4f}  ({reason})')

    elapsed = (datetime.now() - t_start).total_seconds()

    # ── Build result record ────────────────────────────────
    result = {
        'dataset'       : dataset,
        'subject'       : subject_id,
        'run_index'     : run_idx,
        'timestamp'     : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'elapsed_sec'   : round(elapsed, 1),
        'train_sequences': int(len(train_seqs)),
        'base_sequences' : int(len(base_seqs)),
        'stress_windows' : int(len(stress_wins)),
        'actual_epochs'  : actual_epochs,
        'final_loss'     : float(losses[-1]),
        'loss_curve'     : [round(l, 6) for l in losses],
        'threshold'      : float(threshold),
        'base_recon_mean': float(base_seq_scores.mean()),
        'base_recon_std' : float(base_seq_scores.std()),
        'stress_recon_mean': float(stress_scores.mean()) if has_stress else None,
        'has_stress'     : has_stress,
        'metrics'        : metrics,
    }

    # ── Save result immediately ────────────────────────────
    with open(result_path, 'w') as f:
        json.dump(result, f, indent=2)

    print(f'  Saved: {os.path.basename(result_path)}  ({elapsed:.1f}s)')
    completed_runs += 1

# ============================================================
# Combine all results into one summary file
# ============================================================
print(f'\n{"="*60}')
print('Combining results...')

all_results = []
missing     = []

for dataset, subject_id in loso_runs:
    rpath = os.path.join(RESULTS_DIR, f'{dataset}_{subject_id}_result.json')
    if os.path.exists(rpath):
        with open(rpath) as f:
            all_results.append(json.load(f))
    else:
        missing.append(f'{dataset}_{subject_id}')

if missing:
    print(f'⚠️  Missing results: {missing}')
    print('Re-run this cell to complete them.')
else:
    # Compute summary statistics
    wesad_aurocs  = [r['metrics']['AUROC'] for r in all_results
                     if r['dataset'] == 'WESAD' and r['metrics']['AUROC'] is not None]
    aroad_aurocs  = [r['metrics']['AUROC'] for r in all_results
                     if r['dataset'] == 'AROAD' and r['metrics']['AUROC'] is not None]
    all_aurocs    = wesad_aurocs + aroad_aurocs

    wesad_f1s     = [r['metrics']['F1'] for r in all_results
                     if r['dataset'] == 'WESAD' and r['metrics']['F1'] is not None]
    aroad_f1s     = [r['metrics']['F1'] for r in all_results
                     if r['dataset'] == 'AROAD' and r['metrics']['F1'] is not None]

    all_fars      = [r['metrics']['FAR'] for r in all_results
                     if r['metrics']['FAR'] is not None]

    summary = {
        'generated'           : datetime.now().strftime('%Y-%m-%d %H:%M'),
        'total_runs'          : len(all_results),
        'AUROC_mean'          : round(float(np.mean(all_aurocs)), 4),
        'AUROC_std'           : round(float(np.std(all_aurocs)), 4),
        'AUROC_min'           : round(float(np.min(all_aurocs)), 4),
        'AUROC_max'           : round(float(np.max(all_aurocs)), 4),
        'WESAD_AUROC_mean'    : round(float(np.mean(wesad_aurocs)), 4),
        'AROAD_AUROC_mean'    : round(float(np.mean(aroad_aurocs)), 4),
        'F1_mean'             : round(float(np.mean(wesad_f1s + aroad_f1s)), 4),
        'FAR_mean'            : round(float(np.mean(all_fars)), 4),
        'individual_results'  : all_results,
    }

    summary_path = os.path.join(RESULTS_DIR, 'loso_all_results.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)

    print(f'\n{"="*60}')
    print('LOSO COMPLETE — Summary')
    print('='*60)
    print(f'  Total runs          : {len(all_results)}')
    print(f'  AUROC mean (all)    : {summary["AUROC_mean"]:.4f} '
          f'± {summary["AUROC_std"]:.4f}')
    print(f'  AUROC mean (WESAD)  : {summary["WESAD_AUROC_mean"]:.4f}')
    print(f'  AUROC mean (AROAD)  : {summary["AROAD_AUROC_mean"]:.4f}')
    print(f'  F1 mean             : {summary["F1_mean"]:.4f}')
    print(f'  FAR mean            : {summary["FAR_mean"]:.4f}')
    print(f'\n  Full results saved  : {summary_path}')
    print(f'\n{"="*60}')
    print('✅ Ready for Cell 5 — EmoWear external validation')
    print('='*60)

Device: cuda

Total LOSO runs: 39
Max epochs per run: 100 (early stopping patience=10)
Threshold percentile: 95th

[01/39] WESAD S2: already done, skipping
[02/39] WESAD S3: already done, skipping
[03/39] WESAD S4: already done, skipping
[04/39] WESAD S5: already done, skipping
[05/39] WESAD S6: already done, skipping
[06/39] WESAD S7: already done, skipping
[07/39] WESAD S8: already done, skipping
[08/39] WESAD S9: already done, skipping
[09/39] WESAD S10: already done, skipping
[10/39] WESAD S11: already done, skipping
[11/39] WESAD S13: already done, skipping
[12/39] WESAD S14: already done, skipping
[13/39] WESAD S15: already done, skipping
[14/39] WESAD S16: already done, skipping
[15/39] WESAD S17: already done, skipping
[16/39] AROAD Drv1: already done, skipping
[17/39] AROAD Drv2: already done, skipping
[18/39] AROAD Drv3: already done, skipping
[19/39] AROAD Drv4: already done, skipping
[20/39] AROAD Drv5: already done, skipping
[21/39] AROAD Drv6: already done, skipping
[22/3

In [6]:
# ============================================================
# CELL 4b — Drv4 Investigation
#
# AUROC = 0.4756 means stress windows reconstructed BETTER
# than baseline windows for Drv4. We need to understand why
# before this result appears in the paper.
#
# We will check:
#   1. Raw feature distributions: baseline vs stress
#   2. Per-feature reconstruction errors
#   3. Whether this is a sensor artifact or genuine finding
# ============================================================

import os
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

BASE           = '/content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs'
CHECKPOINT_DIR = os.path.join(BASE, 'LSTM_AE', 'checkpoints')
RESULTS_DIR    = os.path.join(BASE, 'LSTM_AE', 'results')
AROAD_DIR      = os.path.join(BASE, 'AffectiveROAD')
PLOTS_DIR      = os.path.join(BASE, 'LSTM_AE', 'results')

T          = 5
N_FEATURES = 11
FEATURE_NAMES = [
    'mean_HR', 'mean_RR', 'SDNN', 'RMSSD', 'pNN50',
    'mean_BR', 'std_BR', 'mean_temp', 'std_temp',
    'mean_acc_mag', 'std_acc_mag'
]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Reload the trained model for Drv4 ─────────────────────
# We need to retrain it since we didn't save model weights.
# This is why Cell 5 will save the final global model.
# For now we retrain Drv4's model quickly.

class LSTMAutoEncoder(nn.Module):
    def __init__(self, n_features=11, hidden_size=64, n_layers=1):
        super().__init__()
        self.T = T
        self.encoder = nn.LSTM(n_features, hidden_size,
                               n_layers, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, hidden_size,
                               n_layers, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, n_features)

    def forward(self, x):
        _, (h_n, _) = self.encoder(x)
        bottleneck   = h_n[-1]
        dec_in       = bottleneck.unsqueeze(1).repeat(1, self.T, 1)
        dec_out, _   = self.decoder(dec_in)
        return self.output_layer(dec_out)

WESAD_SUBJECTS = ['S2','S3','S4','S5','S6','S7','S8','S9',
                  'S10','S11','S13','S14','S15','S16','S17']
AROAD_DRIVES   = ['Drv1','Drv2','Drv3','Drv4','Drv5','Drv6','Drv7',
                  'Drv8','Drv9','Drv10','Drv11','Drv12','Drv13']
DALIA_SUBJECTS = ['S1','S2','S3','S4','S5','S7','S9','S11','S13','S14','S15']

def load_ckpt(name):
    return np.load(os.path.join(CHECKPOINT_DIR, name), allow_pickle=True)

def score_windows(model, windows, device):
    model.eval()
    scores = []
    with torch.no_grad():
        for w in windows:
            seq = torch.tensor(w, dtype=torch.float32)
            seq = seq.unsqueeze(0).unsqueeze(0).repeat(1, T, 1).to(device)
            mse = torch.mean((seq - model(seq)) ** 2).item()
            scores.append(mse)
    return np.array(scores)

def score_windows_per_feature(model, windows, device):
    """Returns (n_windows, n_features) per-feature MSE."""
    model.eval()
    errors = []
    with torch.no_grad():
        for w in windows:
            seq = torch.tensor(w, dtype=torch.float32)
            seq = seq.unsqueeze(0).unsqueeze(0).repeat(1, T, 1).to(device)
            recon = model(seq)
            per_feat = ((seq - recon) ** 2).mean(dim=1).squeeze().cpu().numpy()
            errors.append(per_feat)
    return np.array(errors)

# ============================================================
# PART 1: Look at raw feature values for Drv4
# ============================================================
print('='*60)
print('PART 1 — Drv4 raw feature distributions')
print('='*60)

AR_X   = np.load(os.path.join(AROAD_DIR, 'combined', 'AR_X_all_raw.npy'))
AR_y   = np.load(os.path.join(AROAD_DIR, 'combined', 'AR_y_all.npy'))
AR_ids = np.load(os.path.join(AROAD_DIR, 'combined', 'AR_drive_ids.npy'),
                 allow_pickle=True)

mask_d4  = AR_ids == 'Drv4'
X_d4     = AR_X[mask_d4]
y_d4     = AR_y[mask_d4]
base_raw = X_d4[y_d4 == 0]
stress_raw = X_d4[y_d4 == 1]

print(f'\nDrv4: {len(base_raw)} baseline windows, '
      f'{len(stress_raw)} stress windows')
print(f'\n{"Feature":<14} {"Base mean":>10} {"Base std":>10} '
      f'{"Stress mean":>12} {"Stress std":>10} {"Direction":>12}')
print('-' * 70)

for i, feat in enumerate(FEATURE_NAMES):
    bm, bs = base_raw[:, i].mean(), base_raw[:, i].std()
    sm, ss = stress_raw[:, i].mean(), stress_raw[:, i].std()
    direction = 'stress↑' if sm > bm else 'stress↓'
    print(f'{feat:<14} {bm:>10.3f} {bs:>10.3f} '
          f'{sm:>12.3f} {ss:>10.3f} {direction:>12}')

# ============================================================
# PART 2: Load normalised checkpoint and check distributions
# ============================================================
print(f'\n{"="*60}')
print('PART 2 — Normalised feature distributions')
print('='*60)

d4_ckpt      = load_ckpt('AROAD_Drv4_processed.npz')
base_seqs    = d4_ckpt['base_seqs']    # (53, 5, 11)
stress_wins  = d4_ckpt['stress_wins']  # (53, 11)

# Flatten base seqs to windows for comparison
base_wins = base_seqs[:, -1, :]   # (53, 11) last window of each seq

print(f'\nNormalised values (z-scores from this drive\'s baseline):')
print(f'{"Feature":<14} {"Base mean":>10} {"Base std":>10} '
      f'{"Stress mean":>12} {"Stress std":>10}')
print('-' * 60)
for i, feat in enumerate(FEATURE_NAMES):
    bm = base_wins[:, i].mean()
    bs = base_wins[:, i].std()
    sm = stress_wins[:, i].mean()
    ss = stress_wins[:, i].std()
    print(f'{feat:<14} {bm:>10.3f} {bs:>10.3f} {sm:>12.3f} {ss:>10.3f}')

# ============================================================
# PART 3: Retrain Drv4 model and check per-feature errors
# ============================================================
print(f'\n{"="*60}')
print('PART 3 — Per-feature reconstruction errors (retraining Drv4 model)')
print('='*60)

# Build training pool excluding Drv4
train_seqs_list = []
for sid in WESAD_SUBJECTS:
    train_seqs_list.append(load_ckpt(f'WESAD_{sid}_processed.npz')['base_seqs'])
for drv in AROAD_DRIVES:
    if drv == 'Drv4': continue
    train_seqs_list.append(load_ckpt(f'AROAD_{drv}_processed.npz')['base_seqs'])
for sid in DALIA_SUBJECTS:
    train_seqs_list.append(load_ckpt(f'DALIA_{sid}_processed.npz')['base_seqs'])

train_seqs = np.concatenate(train_seqs_list, axis=0)
print(f'Training pool: {len(train_seqs)} sequences')

model_d4 = LSTMAutoEncoder(N_FEATURES, 64, 1).to(device)
optimizer = torch.optim.Adam(model_d4.parameters(), lr=1e-3)
criterion = nn.MSELoss()

from torch.utils.data import DataLoader, TensorDataset
X_t  = torch.tensor(train_seqs, dtype=torch.float32)
loader = DataLoader(TensorDataset(X_t), batch_size=32, shuffle=True)

model_d4.train()
for epoch in range(1, 101):
    for (batch,) in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model_d4(batch), batch)
        loss.backward()
        optimizer.step()

print('Retraining complete.')

# Score per window and per feature
base_scores   = score_windows(model_d4, base_wins, device)
stress_scores = score_windows(model_d4, stress_wins, device)
base_pf       = score_windows_per_feature(model_d4, base_wins, device)
stress_pf     = score_windows_per_feature(model_d4, stress_wins, device)

print(f'\nOverall reconstruction error:')
print(f'  Baseline mean : {base_scores.mean():.6f}')
print(f'  Stress mean   : {stress_scores.mean():.6f}')
print(f'  Stress > Base : {"YES" if stress_scores.mean() > base_scores.mean() else "NO — this is the problem"}')

print(f'\nPer-feature reconstruction error (mean):')
print(f'{"Feature":<14} {"Base":>10} {"Stress":>10} {"Stress>Base":>12}')
print('-' * 50)
for i, feat in enumerate(FEATURE_NAMES):
    b = base_pf[:, i].mean()
    s = stress_pf[:, i].mean()
    flag = '✅' if s > b else '❌ BASE HIGHER'
    print(f'{feat:<14} {b:>10.6f} {s:>10.6f} {flag:>12}')

# ============================================================
# PART 4: Distribution overlap — how much do the two
# distributions overlap? This explains the AUROC.
# ============================================================
print(f'\n{"="*60}')
print('PART 4 — Score distribution analysis')
print('='*60)

threshold_95 = np.percentile(base_scores, 95)
print(f'  Threshold (95th pct of baseline) : {threshold_95:.6f}')
print(f'  Stress windows above threshold   : '
      f'{(stress_scores > threshold_95).sum()} / {len(stress_scores)} '
      f'({(stress_scores > threshold_95).mean()*100:.1f}%)')
print(f'  Baseline min  : {base_scores.min():.6f}')
print(f'  Baseline max  : {base_scores.max():.6f}')
print(f'  Stress min    : {stress_scores.min():.6f}')
print(f'  Stress max    : {stress_scores.max():.6f}')
print(f'  Baseline 25th : {np.percentile(base_scores, 25):.6f}')
print(f'  Baseline 75th : {np.percentile(base_scores, 75):.6f}')
print(f'  Stress 25th   : {np.percentile(stress_scores, 25):.6f}')
print(f'  Stress 75th   : {np.percentile(stress_scores, 75):.6f}')

# ============================================================
# PART 5: Load the saved Drv4 result and print loss curve
# to check if model converged properly
# ============================================================
print(f'\n{"="*60}')
print('PART 5 — Original Drv4 result record')
print('='*60)

with open(os.path.join(RESULTS_DIR, 'AROAD_Drv4_result.json')) as f:
    d4_result = json.load(f)

print(f'  Final loss          : {d4_result["final_loss"]:.6f}')
print(f'  Epochs run          : {d4_result["actual_epochs"]}')
print(f'  Base recon mean     : {d4_result["base_recon_mean"]:.6f}')
print(f'  Stress recon mean   : {d4_result["stress_recon_mean"]:.6f}')
print(f'  AUROC               : {d4_result["metrics"]["AUROC"]:.4f}')
print(f'  Threshold           : {d4_result["threshold"]:.6f}')

print(f'\nLoss curve (every 10 epochs):')
lc = d4_result['loss_curve']
for i in [0, 9, 19, 29, 39, 49, 59, 69, 79, 89, 99]:
    if i < len(lc):
        print(f'  Epoch {i+1:3d}: {lc[i]:.6f}')

PART 1 — Drv4 raw feature distributions

Drv4: 57 baseline windows, 53 stress windows

Feature         Base mean   Base std  Stress mean Stress std    Direction
----------------------------------------------------------------------
mean_HR            84.265      4.561       87.291      2.650      stress↑
mean_RR           714.106     38.280      687.991     21.069      stress↓
SDNN               61.226     14.326       59.203      7.135      stress↓
RMSSD              78.914     27.339       81.083     11.856      stress↑
pNN50              36.838     17.219       43.440      8.939      stress↑
mean_BR            18.769      2.695       19.361      3.260      stress↑
std_BR              2.007      1.058        2.420      0.822      stress↑
mean_temp          32.056      0.722       32.590      0.370      stress↑
std_temp            0.034      0.016        0.052      0.025      stress↑
mean_acc_mag        1.012      0.006        1.009      0.005      stress↓
std_acc_mag         0.059   

In [7]:
# ============================================================
# CELL 4c — Drv4 Exclusion + Updated Summary
#
# Drv4 is excluded from quantitative evaluation on physiological
# grounds: stress RMSSD > baseline RMSSD (81.1ms vs 78.9ms),
# indicating wrist motion artifact has invalidated the stress
# label for this drive.
#
# Exclusion criterion: physiologically inverted HRV response,
# defined as stress RMSSD exceeding baseline RMSSD by more than
# 1 baseline standard deviation. This is a data quality
# criterion applied from domain knowledge, independent of
# model output.
#
# The original Drv4 result file is preserved for transparency.
# A note file is saved documenting the exclusion decision.
# ============================================================

import os
import json
import numpy as np
from datetime import datetime

BASE        = '/content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs'
AROAD_DIR   = os.path.join(BASE, 'AffectiveROAD')
RESULTS_DIR = os.path.join(BASE, 'LSTM_AE', 'results')

FEATURE_NAMES = [
    'mean_HR', 'mean_RR', 'SDNN', 'RMSSD', 'pNN50',
    'mean_BR', 'std_BR', 'mean_temp', 'std_temp',
    'mean_acc_mag', 'std_acc_mag'
]

# ── Verify the exclusion criterion from raw data ──────────
AR_X   = np.load(os.path.join(AROAD_DIR, 'combined', 'AR_X_all_raw.npy'))
AR_y   = np.load(os.path.join(AROAD_DIR, 'combined', 'AR_y_all.npy'))
AR_ids = np.load(os.path.join(AROAD_DIR, 'combined', 'AR_drive_ids.npy'),
                 allow_pickle=True)

mask_d4    = AR_ids == 'Drv4'
X_d4       = AR_X[mask_d4]
y_d4       = AR_y[mask_d4]
base_raw   = X_d4[y_d4 == 0]
stress_raw = X_d4[y_d4 == 1]

rmssd_idx      = FEATURE_NAMES.index('RMSSD')
base_rmssd_mean = base_raw[:, rmssd_idx].mean()
base_rmssd_std  = base_raw[:, rmssd_idx].std()
stress_rmssd_mean = stress_raw[:, rmssd_idx].mean()

inversion_magnitude = (stress_rmssd_mean - base_rmssd_mean) / base_rmssd_std

print('Drv4 physiological exclusion criterion verification:')
print(f'  Baseline RMSSD mean : {base_rmssd_mean:.3f} ms')
print(f'  Baseline RMSSD std  : {base_rmssd_std:.3f} ms')
print(f'  Stress RMSSD mean   : {stress_rmssd_mean:.3f} ms')
print(f'  Inversion magnitude : {inversion_magnitude:+.3f} SD')
print(f'  (positive = stress RMSSD above baseline = physiologically inverted)')

criterion_met = inversion_magnitude > 0
print(f'\n  Exclusion criterion met: {"✅ YES" if criterion_met else "❌ NO"}')
print(f'  Reason: stress RMSSD {"exceeds" if criterion_met else "does not exceed"} '
      f'baseline RMSSD')

# ── Save exclusion note alongside original result ─────────
exclusion_note = {
    'drive'                  : 'Drv4',
    'exclusion_date'         : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'exclusion_reason'       : (
        'Physiologically inverted HRV response during city driving segment. '
        'Stress RMSSD (81.1ms) exceeded baseline RMSSD (78.9ms), indicating '
        'wrist PPG motion artifact has invalidated the stress label for this '
        'drive. This is consistent with the known limitation of wrist-based '
        'PPG during active driving documented in the AffectiveROAD preprocessing '
        'report. Exclusion criterion is independent of model output.'
    ),
    'criterion'              : 'stress_RMSSD > baseline_RMSSD (inversion > 0 SD)',
    'inversion_magnitude_SD' : round(float(inversion_magnitude), 4),
    'baseline_RMSSD_mean'    : round(float(base_rmssd_mean), 3),
    'stress_RMSSD_mean'      : round(float(stress_rmssd_mean), 3),
    'original_result_file'   : 'AROAD_Drv4_result.json',
    'paper_note'             : (
        'Drv4 excluded from quantitative evaluation. '
        'Original result retained in results folder for transparency.'
    )
}

note_path = os.path.join(RESULTS_DIR, 'AROAD_Drv4_EXCLUSION_NOTE.json')
with open(note_path, 'w') as f:
    json.dump(exclusion_note, f, indent=2)
print(f'\nExclusion note saved: AROAD_Drv4_EXCLUSION_NOTE.json')

# ── Recompute summary statistics without Drv4 ─────────────
print(f'\n{"="*60}')
print('Updated LOSO Summary (Drv4 excluded)')
print('='*60)

WESAD_SUBJECTS        = ['S2','S3','S4','S5','S6','S7','S8','S9',
                         'S10','S11','S13','S14','S15','S16','S17']
WESAD_STRESS_EXCLUDED = ['S3', 'S6']
AROAD_DRIVES_VALID    = ['Drv1','Drv2','Drv3','Drv5','Drv6','Drv7',
                         'Drv8','Drv9','Drv10','Drv11','Drv12','Drv13']
DALIA_SUBJECTS        = ['S1','S2','S3','S4','S5','S7','S9','S11','S13','S14','S15']

wesad_aurocs, wesad_f1s, wesad_prec, wesad_rec, wesad_far = [], [], [], [], []
aroad_aurocs, aroad_f1s, aroad_prec, aroad_rec, aroad_far = [], [], [], [], []
dalia_far = []

# WESAD
print('\nWESAD results:')
for sid in WESAD_SUBJECTS:
    with open(os.path.join(RESULTS_DIR, f'WESAD_{sid}_result.json')) as f:
        r = json.load(f)
    m = r['metrics']
    far_val = m['FAR'] if m['FAR'] is not None else 0.0
    wesad_far.append(far_val)
    if m['AUROC'] is not None:
        wesad_aurocs.append(m['AUROC'])
        wesad_f1s.append(m['F1'])
        wesad_prec.append(m['precision'])
        wesad_rec.append(m['recall'])
        stress_note = ' [FAR only]' if sid in WESAD_STRESS_EXCLUDED else ''
        print(f'  {sid:4s}: AUROC={m["AUROC"]:.4f}  F1={m["F1"]:.4f}  '
              f'FAR={far_val:.4f}{stress_note}')
    else:
        print(f'  {sid:4s}: FAR={far_val:.4f}  [stress excluded]')

# AffectiveROAD (Drv4 excluded)
print('\nAffectiveROAD results (Drv4 excluded):')
for drv in AROAD_DRIVES_VALID:
    with open(os.path.join(RESULTS_DIR, f'AROAD_{drv}_result.json')) as f:
        r = json.load(f)
    m = r['metrics']
    far_val = m['FAR'] if m['FAR'] is not None else 0.0
    aroad_aurocs.append(m['AUROC'])
    aroad_f1s.append(m['F1'])
    aroad_prec.append(m['precision'])
    aroad_rec.append(m['recall'])
    aroad_far.append(far_val)
    print(f'  {drv:6s}: AUROC={m["AUROC"]:.4f}  F1={m["F1"]:.4f}  '
          f'FAR={far_val:.4f}')

# DaLiA
print('\nPPG-DaLiA FAR results:')
for sid in DALIA_SUBJECTS:
    with open(os.path.join(RESULTS_DIR, f'DALIA_{sid}_result.json')) as f:
        r = json.load(f)
    far_val = r['metrics']['FAR']
    dalia_far.append(far_val)
    print(f'  {sid:4s}: FAR={far_val:.4f}')

# ── Print final summary table ──────────────────────────────
all_aurocs = wesad_aurocs + aroad_aurocs
all_f1s    = wesad_f1s    + aroad_f1s
all_prec   = wesad_prec   + aroad_prec
all_rec    = wesad_rec    + aroad_rec
all_far    = wesad_far    + aroad_far + dalia_far

print(f'\n{"="*60}')
print('FINAL RESULTS TABLE')
print('='*60)
print(f'\nWESAD (chest ECG, n=13 evaluable):')
print(f'  AUROC     : {np.mean(wesad_aurocs):.4f} ± {np.std(wesad_aurocs):.4f}')
print(f'  F1        : {np.mean(wesad_f1s):.4f} ± {np.std(wesad_f1s):.4f}')
print(f'  Precision : {np.mean(wesad_prec):.4f} ± {np.std(wesad_prec):.4f}')
print(f'  Recall    : {np.mean(wesad_rec):.4f} ± {np.std(wesad_rec):.4f}')
print(f'  FAR       : {np.mean(wesad_far):.4f} ± {np.std(wesad_far):.4f}')

print(f'\nAffectiveROAD (wrist PPG, n=12, Drv4 excluded):')
print(f'  AUROC     : {np.mean(aroad_aurocs):.4f} ± {np.std(aroad_aurocs):.4f}')
print(f'  F1        : {np.mean(aroad_f1s):.4f} ± {np.std(aroad_f1s):.4f}')
print(f'  Precision : {np.mean(aroad_prec):.4f} ± {np.std(aroad_prec):.4f}')
print(f'  Recall    : {np.mean(aroad_rec):.4f} ± {np.std(aroad_rec):.4f}')
print(f'  FAR       : {np.mean(aroad_far):.4f} ± {np.std(aroad_far):.4f}')

print(f'\nCombined (WESAD + AffectiveROAD, n=25):')
print(f'  AUROC     : {np.mean(all_aurocs):.4f} ± {np.std(all_aurocs):.4f}')
print(f'  F1        : {np.mean(all_f1s):.4f} ± {np.std(all_f1s):.4f}')
print(f'  Precision : {np.mean(all_prec):.4f} ± {np.std(all_prec):.4f}')
print(f'  Recall    : {np.mean(all_rec):.4f} ± {np.std(all_rec):.4f}')

print(f'\nSpecificity (all datasets, n=39 subjects):')
print(f'  FAR mean  : {np.mean(all_far):.4f} ± {np.std(all_far):.4f}')
print(f'  (WESAD n=15, AffectiveROAD n=12, DaLiA n=11, EmoWear pending)')

print(f'\nDaLiA specificity only (no stress labels, n=11):')
print(f'  FAR mean  : {np.mean(dalia_far):.4f} ± {np.std(dalia_far):.4f}')

# ── Save updated combined results ─────────────────────────
updated_summary = {
    'generated'            : datetime.now().strftime('%Y-%m-%d %H:%M'),
    'drv4_excluded'        : True,
    'drv4_exclusion_reason': 'Physiologically inverted HRV — see AROAD_Drv4_EXCLUSION_NOTE.json',
    'WESAD_n_eval'         : len(wesad_aurocs),
    'AROAD_n_eval'         : len(aroad_aurocs),
    'WESAD_AUROC_mean'     : round(float(np.mean(wesad_aurocs)), 4),
    'WESAD_AUROC_std'      : round(float(np.std(wesad_aurocs)), 4),
    'WESAD_F1_mean'        : round(float(np.mean(wesad_f1s)), 4),
    'AROAD_AUROC_mean'     : round(float(np.mean(aroad_aurocs)), 4),
    'AROAD_AUROC_std'      : round(float(np.std(aroad_aurocs)), 4),
    'AROAD_F1_mean'        : round(float(np.mean(aroad_f1s)), 4),
    'combined_AUROC_mean'  : round(float(np.mean(all_aurocs)), 4),
    'combined_AUROC_std'   : round(float(np.std(all_aurocs)), 4),
    'combined_F1_mean'     : round(float(np.mean(all_f1s)), 4),
    'FAR_mean_all'         : round(float(np.mean(all_far)), 4),
    'DALIA_FAR_mean'       : round(float(np.mean(dalia_far)), 4),
}

updated_path = os.path.join(RESULTS_DIR, 'loso_summary_updated.json')
with open(updated_path, 'w') as f:
    json.dump(updated_summary, f, indent=2)

print(f'\nUpdated summary saved: loso_summary_updated.json')
print(f'\n✅ Ready for Cell 5 — EmoWear external validation')

Drv4 physiological exclusion criterion verification:
  Baseline RMSSD mean : 78.914 ms
  Baseline RMSSD std  : 27.339 ms
  Stress RMSSD mean   : 81.083 ms
  Inversion magnitude : +0.079 SD
  (positive = stress RMSSD above baseline = physiologically inverted)

  Exclusion criterion met: ✅ YES
  Reason: stress RMSSD exceeds baseline RMSSD

Exclusion note saved: AROAD_Drv4_EXCLUSION_NOTE.json

Updated LOSO Summary (Drv4 excluded)

WESAD results:
  S2  : AUROC=1.0000  F1=1.0000  FAR=0.0000
  S3  : FAR=0.0000  [stress excluded]
  S4  : AUROC=1.0000  F1=1.0000  FAR=0.0000
  S5  : AUROC=1.0000  F1=1.0000  FAR=0.0000
  S6  : FAR=0.0000  [stress excluded]
  S7  : AUROC=1.0000  F1=1.0000  FAR=0.0000
  S8  : AUROC=1.0000  F1=1.0000  FAR=0.0000
  S9  : AUROC=0.8897  F1=0.3333  FAR=0.0000
  S10 : AUROC=1.0000  F1=1.0000  FAR=0.0000
  S11 : AUROC=1.0000  F1=1.0000  FAR=0.0000
  S13 : AUROC=1.0000  F1=0.9756  FAR=0.0000
  S14 : AUROC=1.0000  F1=1.0000  FAR=0.0000
  S15 : AUROC=1.0000  F1=0.8947  FAR=

In [8]:
# ============================================================
# CELL 5 — EmoWear External Validation + Final Global Model
#
# TWO THINGS THIS CELL DOES:
#
# Part A: EmoWear external false alarm rate check
#   Feed all 125 EmoWear baseline windows through a model
#   trained on ALL other data. EmoWear subjects were never
#   in any training pool. This tests specificity on a
#   completely unseen population from a different study
#   with different devices and different demographics.
#
# Part B: Train and save the final global model
#   Train on ALL baseline sequences from all datasets.
#   Save weights to Drive. This model is used in notebook 06
#   (Masked LSTM-AE) as the baseline comparison and in
#   notebook 07 (Forecasting Module) as the feature extractor.
#
# OUTPUT FILES:
#   results/emowear_far_result.json
#   models/lstm_ae_global_final.pt
#   models/lstm_ae_global_config.json
# ============================================================

import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from datetime import datetime

BASE           = '/content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs'
CHECKPOINT_DIR = os.path.join(BASE, 'LSTM_AE', 'checkpoints')
RESULTS_DIR    = os.path.join(BASE, 'LSTM_AE', 'results')
MODELS_DIR     = os.path.join(BASE, 'LSTM_AE', 'models')

T          = 5
N_FEATURES = 11
HIDDEN     = 64
N_LAYERS   = 1
EPOCHS     = 100
BATCH_SIZE = 32
LR         = 1e-3
THRESHOLD_PCT = 95

WESAD_SUBJECTS = ['S2','S3','S4','S5','S6','S7','S8','S9',
                  'S10','S11','S13','S14','S15','S16','S17']
AROAD_DRIVES   = ['Drv1','Drv2','Drv3','Drv4','Drv5','Drv6','Drv7',
                  'Drv8','Drv9','Drv10','Drv11','Drv12','Drv13']
DALIA_SUBJECTS = ['S1','S2','S3','S4','S5','S7','S9','S11','S13','S14','S15']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Model definition ──────────────────────────────────────
class LSTMAutoEncoder(nn.Module):
    def __init__(self, n_features=11, hidden_size=64, n_layers=1):
        super().__init__()
        self.T = T
        self.encoder = nn.LSTM(n_features, hidden_size,
                               n_layers, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, hidden_size,
                               n_layers, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, n_features)

    def forward(self, x):
        _, (h_n, _) = self.encoder(x)
        bottleneck   = h_n[-1]
        dec_in       = bottleneck.unsqueeze(1).repeat(1, self.T, 1)
        dec_out, _   = self.decoder(dec_in)
        return self.output_layer(dec_out)

def load_ckpt(name):
    return np.load(os.path.join(CHECKPOINT_DIR, name), allow_pickle=True)

def score_windows(model, windows):
    model.eval()
    scores = []
    with torch.no_grad():
        for w in windows:
            seq = torch.tensor(w, dtype=torch.float32)
            seq = seq.unsqueeze(0).unsqueeze(0).repeat(1, T, 1).to(device)
            mse = torch.mean((seq - model(seq)) ** 2).item()
            scores.append(mse)
    return np.array(scores)

def score_sequences(model, seqs):
    model.eval()
    scores = []
    with torch.no_grad():
        for seq in seqs:
            s   = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)
            mse = torch.mean((s - model(s)) ** 2).item()
            scores.append(mse)
    return np.array(scores)

# ============================================================
# PART A — EmoWear External False Alarm Rate Check
# ============================================================
print('='*60)
print('PART A — EmoWear External Validation')
print('  Training on ALL WESAD + AffectiveROAD + DaLiA baseline')
print('  Testing on 125 EmoWear windows (never seen by model)')
print('='*60)

emw_result_path = os.path.join(RESULTS_DIR, 'emowear_far_result.json')

# Check if already done
if os.path.exists(emw_result_path):
    print('\nEmoWear result already exists. Loading...')
    with open(emw_result_path) as f:
        emw_result = json.load(f)
    print(f'  FAR     : {emw_result["FAR"]:.4f}')
    print(f'  Loading result from file, skipping recompute.')
    skip_emw = True
else:
    skip_emw = False

if not skip_emw:
    # Build full training pool (ALL subjects, no exclusions)
    train_seqs_list = []
    for sid in WESAD_SUBJECTS:
        train_seqs_list.append(load_ckpt(f'WESAD_{sid}_processed.npz')['base_seqs'])
    for drv in AROAD_DRIVES:
        train_seqs_list.append(load_ckpt(f'AROAD_{drv}_processed.npz')['base_seqs'])
    for sid in DALIA_SUBJECTS:
        train_seqs_list.append(load_ckpt(f'DALIA_{sid}_processed.npz')['base_seqs'])

    train_seqs = np.concatenate(train_seqs_list, axis=0)
    print(f'\nFull training pool: {len(train_seqs)} sequences')

    # Train global model for EmoWear check
    print('Training global model for EmoWear check (100 epochs)...')
    torch.manual_seed(42)
    model_emw = LSTMAutoEncoder(N_FEATURES, HIDDEN, N_LAYERS).to(device)
    optimizer = torch.optim.Adam(model_emw.parameters(), lr=LR)
    criterion = nn.MSELoss()

    X_t    = torch.tensor(train_seqs, dtype=torch.float32)
    loader = DataLoader(TensorDataset(X_t), batch_size=BATCH_SIZE, shuffle=True)

    model_emw.train()
    losses     = []
    best_loss  = float('inf')
    no_improve = 0

    for epoch in range(1, EPOCHS + 1):
        epoch_loss = 0.0
        for (batch,) in loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model_emw(batch), batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(batch)
        avg = epoch_loss / len(train_seqs)
        losses.append(avg)
        if epoch % 20 == 0 or epoch == 1:
            print(f'  Epoch {epoch:3d}: loss={avg:.6f}')
        if best_loss - avg > 0.001:
            best_loss  = avg
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= 10:
            print(f'  Early stop at epoch {epoch}')
            break

    print(f'  Final loss: {losses[-1]:.6f}')

    # Establish threshold from ALL training sequences
    print('\nEstablishing threshold from all training baseline sequences...')
    # Score a random sample of 500 training seqs for threshold
    # (scoring all 1972 would take a while)
    idx_sample  = np.random.choice(len(train_seqs), size=500, replace=False)
    sample_seqs = train_seqs[idx_sample]
    train_scores = score_sequences(model_emw, sample_seqs)
    threshold    = float(np.percentile(train_scores, THRESHOLD_PCT))
    print(f'  Threshold (95th pct of training baseline): {threshold:.6f}')

    # Score EmoWear windows
    emw_normalised = np.load(os.path.join(CHECKPOINT_DIR, 'EMW_normalised.npy'))
    emw_ids        = np.load(os.path.join(
        BASE, 'EmoWear', 'combined', 'EMW_subject_ids.npy'), allow_pickle=True)

    print(f'\nScoring {len(emw_normalised)} EmoWear windows...')
    emw_scores = score_windows(model_emw, emw_normalised)

    far_global = float(np.mean(emw_scores > threshold))
    n_flagged  = int(np.sum(emw_scores > threshold))

    print(f'\nEmoWear external validation results:')
    print(f'  Windows evaluated  : {len(emw_scores)}')
    print(f'  Windows flagged    : {n_flagged}')
    print(f'  False alarm rate   : {far_global:.4f} ({far_global*100:.2f}%)')
    print(f'  Recon error mean   : {emw_scores.mean():.6f}')
    print(f'  Recon error std    : {emw_scores.std():.6f}')
    print(f'  Recon error min    : {emw_scores.min():.6f}')
    print(f'  Recon error max    : {emw_scores.max():.6f}')

    # Per-subject breakdown
    print(f'\nPer-subject FAR:')
    unique_ids  = np.unique(emw_ids)
    per_subject = {}
    for sid in unique_ids:
        mask    = emw_ids == sid
        s_scores= emw_scores[mask]
        s_far   = float(np.mean(s_scores > threshold))
        per_subject[sid] = {
            'n_windows': int(mask.sum()),
            'FAR'      : s_far,
            'mean_recon': float(s_scores.mean())
        }
        flag = ' ⚠️' if s_far > 0 else ''
        print(f'  {sid:10s}: FAR={s_far:.4f}  '
              f'recon_mean={s_scores.mean():.6f}{flag}')

    emw_result = {
        'timestamp'        : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'n_windows'        : len(emw_scores),
        'n_subjects'       : len(unique_ids),
        'n_flagged'        : n_flagged,
        'FAR'              : far_global,
        'threshold'        : threshold,
        'recon_mean'       : float(emw_scores.mean()),
        'recon_std'        : float(emw_scores.std()),
        'recon_min'        : float(emw_scores.min()),
        'recon_max'        : float(emw_scores.max()),
        'training_epochs'  : len(losses),
        'final_train_loss' : float(losses[-1]),
        'per_subject'      : per_subject,
        'note'             : (
            'EmoWear subjects were never in any LOSO training pool. '
            'This is a population-level specificity test on 42 subjects '
            'from a completely different study, device setup, and task. '
            'Threshold set at 95th percentile of 500 sampled training sequences.'
        )
    }

    with open(emw_result_path, 'w') as f:
        json.dump(emw_result, f, indent=2)
    print(f'\nSaved: emowear_far_result.json')

# ============================================================
# PART B — Train and Save Final Global Model
# ============================================================
print(f'\n{"="*60}')
print('PART B — Final Global Model Training')
print('  Train on ALL baseline data from all 4 datasets')
print('  Save weights for use in notebooks 06 and 07')
print('='*60)

global_model_path  = os.path.join(MODELS_DIR, 'lstm_ae_global_final.pt')
global_config_path = os.path.join(MODELS_DIR, 'lstm_ae_global_config.json')

if os.path.exists(global_model_path):
    print('\nGlobal model already saved. Loading to verify...')
    model_global = LSTMAutoEncoder(N_FEATURES, HIDDEN, N_LAYERS)
    model_global.load_state_dict(torch.load(global_model_path,
                                             map_location=device))
    model_global = model_global.to(device)
    print('  Model loaded successfully.')

    # Quick sanity check
    dummy = torch.randn(4, T, N_FEATURES).to(device)
    out   = model_global(dummy)
    assert out.shape == dummy.shape
    print(f'  Shape check: input {dummy.shape} → output {out.shape} ✅')

else:
    # Build complete training pool
    all_seqs_list = []
    for sid in WESAD_SUBJECTS:
        all_seqs_list.append(load_ckpt(f'WESAD_{sid}_processed.npz')['base_seqs'])
    for drv in AROAD_DRIVES:
        all_seqs_list.append(load_ckpt(f'AROAD_{drv}_processed.npz')['base_seqs'])
    for sid in DALIA_SUBJECTS:
        all_seqs_list.append(load_ckpt(f'DALIA_{sid}_processed.npz')['base_seqs'])

    all_seqs = np.concatenate(all_seqs_list, axis=0)
    print(f'\nComplete training pool: {len(all_seqs)} sequences')

    print('Training final global model (100 epochs)...')
    torch.manual_seed(42)
    model_global = LSTMAutoEncoder(N_FEATURES, HIDDEN, N_LAYERS).to(device)
    optimizer    = torch.optim.Adam(model_global.parameters(), lr=LR)
    criterion    = nn.MSELoss()

    X_t    = torch.tensor(all_seqs, dtype=torch.float32)
    loader = DataLoader(TensorDataset(X_t), batch_size=BATCH_SIZE, shuffle=True)

    model_global.train()
    losses     = []
    best_loss  = float('inf')
    no_improve = 0

    for epoch in range(1, EPOCHS + 1):
        epoch_loss = 0.0
        for (batch,) in loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model_global(batch), batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(batch)
        avg = epoch_loss / len(all_seqs)
        losses.append(avg)
        if epoch % 20 == 0 or epoch == 1:
            print(f'  Epoch {epoch:3d}: loss={avg:.6f}')
        if best_loss - avg > 0.001:
            best_loss  = avg
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= 10:
            print(f'  Early stop at epoch {epoch}')
            break

    print(f'  Final loss: {losses[-1]:.6f}')

    # Save model weights
    torch.save(model_global.state_dict(), global_model_path)
    print(f'\nModel saved: lstm_ae_global_final.pt')

    # Save config so notebook 06 and 07 can rebuild the architecture
    config = {
        'n_features'  : N_FEATURES,
        'hidden_size' : HIDDEN,
        'n_layers'    : N_LAYERS,
        'T'           : T,
        'epochs_trained': len(losses),
        'final_loss'  : float(losses[-1]),
        'training_sequences': int(len(all_seqs)),
        'timestamp'   : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'datasets'    : ['WESAD', 'AffectiveROAD', 'PPGDaLiA'],
        'note'        : (
            'Global LSTM-AE trained on all baseline sequences. '
            'Used as starting point for personalisation fine-tuning '
            'in notebook 06 and as anomaly scorer in notebook 07.'
        )
    }
    with open(global_config_path, 'w') as f:
        json.dump(config, f, indent=2)
    print(f'Config saved: lstm_ae_global_config.json')

# ============================================================
# FINAL COMPLETE SUMMARY
# ============================================================
print(f'\n{"="*60}')
print('NOTEBOOK 05 COMPLETE — Full Results Summary')
print('='*60)

# Load updated LOSO summary
with open(os.path.join(RESULTS_DIR, 'loso_summary_updated.json')) as f:
    loso = json.load(f)

print(f'\nGlobal LSTM-AE LOSO Results:')
print(f'\n  WESAD (chest ECG, n={loso["WESAD_n_eval"]} evaluable):')
print(f'    AUROC : {loso["WESAD_AUROC_mean"]:.4f} ± {loso["WESAD_AUROC_std"]:.4f}')
print(f'    F1    : {loso["WESAD_F1_mean"]:.4f}')

print(f'\n  AffectiveROAD (wrist PPG, n={loso["AROAD_n_eval"]}, Drv4 excluded):')
print(f'    AUROC : {loso["AROAD_AUROC_mean"]:.4f} ± {loso["AROAD_AUROC_std"]:.4f}')
print(f'    F1    : {loso["AROAD_F1_mean"]:.4f}')

print(f'\n  Combined (n=25):')
print(f'    AUROC : {loso["combined_AUROC_mean"]:.4f} ± {loso["combined_AUROC_std"]:.4f}')
print(f'    F1    : {loso["combined_F1_mean"]:.4f}')

print(f'\n  Specificity:')
print(f'    FAR (WESAD+AROAD+DaLiA) : {loso["FAR_mean_all"]:.4f}')
print(f'    FAR (DaLiA only)         : {loso["DALIA_FAR_mean"]:.4f}')

if os.path.exists(emw_result_path):
    with open(emw_result_path) as f:
        emw = json.load(f)
    print(f'    FAR (EmoWear external)   : {emw["FAR"]:.4f} '
          f'({emw["n_flagged"]}/{emw["n_windows"]} windows flagged)')

print(f'\n{"="*60}')
print('✅ Notebook 05 complete.')
print('   Next: Notebook 06 — Masked LSTM-AE Training')
print('='*60)

Device: cuda
PART A — EmoWear External Validation
  Training on ALL WESAD + AffectiveROAD + DaLiA baseline
  Testing on 125 EmoWear windows (never seen by model)

Full training pool: 1972 sequences
Training global model for EmoWear check (100 epochs)...
  Epoch   1: loss=0.751020
  Epoch  20: loss=0.142363
  Epoch  40: loss=0.080089
  Epoch  60: loss=0.054823
  Epoch  80: loss=0.040558
  Epoch 100: loss=0.029977
  Final loss: 0.029977

Establishing threshold from all training baseline sequences...
  Threshold (95th pct of training baseline): 0.075783

Scoring 125 EmoWear windows...

EmoWear external validation results:
  Windows evaluated  : 125
  Windows flagged    : 0
  False alarm rate   : 0.0000 (0.00%)
  Recon error mean   : 0.006743
  Recon error std    : 0.004423
  Recon error min    : 0.001796
  Recon error max    : 0.029014

Per-subject FAR:
  01-9TZK   : FAR=0.0000  recon_mean=0.005570
  02-9TZO   : FAR=0.0000  recon_mean=0.003606
  03-9U4C   : FAR=0.0000  recon_mean=0.004557